# Joins Questions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

## Que 01: Material Property Analysis

**Difficulty:** Easy

### Problem

Attach material metadata to every lab experiment, even when the catalog is incomplete.

You are a data engineer at a materials science lab. Experiment results land in one table while the material catalog lives in another, and the catalog is known to be incomplete: some experiments reference materials that were never registered. The lab lead wants a full experiment report with material details attached wherever they exist.

Write a query that joins `me_experiments` to `me_materials` on `material_id`. Every experiment must appear exactly once in the result: if an experiment's `material_id` has no match in `me_materials`, still return the experiment row with NULL `material_name` and `material_type`. Materials that have no experiments must not appear at all. Return `experiment_date` formatted as text in YYYY-MM-DD form. Sort the results by `experiment_id` in ascending order.

**Schema columns:** `me_experiments.experiment_id`, `me_experiments.material_id`, `me_experiments.experiment_date`, `me_experiments.experiment_results`, `me_materials.material_id`, `me_materials.material_name`, `me_materials.material_type`

**Output columns:** `experiment_id`, `material_id`, `experiment_date`, `experiment_results`, `material_name`, `material_type`

Order the result by `experiment_id` ascending.

### Examples

#### Example 1

**Input:**

**me_experiments:**

| experiment_id | material_id | experiment_date | experiment_results |
|--------------:|------------:|-----------------|-------------------:|
| 1 | 101 | 2023-07-01 | 7.6 |
| 2 | 102 | 2023-07-02 | 8.3 |
| 3 | 103 | 2023-07-03 | 6.9 |
| 4 | 101 | 2023-07-04 | 7.2 |

**me_materials:**

| material_id | material_name | material_type |
|------------:|---------------|---------------|
| 101 | Material A | Type X |
| 102 | Material B | Type Y |
| 104 | Material C | Type Z |

**Output:**

| experiment_id | material_id | experiment_date | experiment_results | material_name | material_type |
|--------------:|------------:|-----------------|-------------------:|---------------|---------------|
| 1 | 101 | 2023-07-01 | 7.6 | Material A | Type X |
| 2 | 102 | 2023-07-02 | 8.3 | Material B | Type Y |
| 3 | 103 | 2023-07-03 | 6.9 | NULL | NULL |
| 4 | 101 | 2023-07-04 | 7.2 | Material A | Type X |

**Explanation:** Experiment 3 references material 103, which is missing from the catalog, so it is kept with NULL `material_name` and `material_type`. Material C (104) has never been used in an experiment, so it does not appear. All four experiments are returned in ascending `experiment_id` order.

### Constraints

- Every experiment appears exactly once; experiments with an unregistered `material_id` keep NULL `material_name` and `material_type`.
- Materials with no experiments must not appear in the output.
- `experiment_date` must be returned as text formatted YYYY-MM-DD.
- Output columns must be exactly `experiment_id`, `material_id`, `experiment_date`, `experiment_results`, `material_name`, `material_type`.
- Sort results by `experiment_id` in ascending order.

In [0]:
me_experiments_data=[(1,101,"2023-07-01",7.6),(2,102,"2023-07-02",8.3),(3,103,"2023-07-03",6.9),(4,101,"2023-07-04",7.2)]

me_experiments_df=spark.createDataFrame(me_experiments_data,["experiment_id","material_id","experiment_date","experiment_results"])

display(me_experiments_df)

me_materials_data=[(101,"Material A","Type X"),(102,"Material B","Type Y"),(104,"Material C","Type Z")]

me_materials_df=spark.createDataFrame(me_materials_data,["material_id","material_name","material_type"])

display(me_materials_df)


# experiment_id	material_id	experiment_date	experiment_results	material_name	material_type

joined_df = me_experiments_df.alias("e").join(me_materials_df.alias("m"), on=col("e.material_id") == col("m.material_id"), how="left").withColumn("experiment_date", date_format("e.experiment_date", "yyyy-MM-dd")).orderBy("e.experiment_id")

display(joined_df)



experiment_id,material_id,experiment_date,experiment_results
1,101,2023-07-01,7.6
2,102,2023-07-02,8.3
3,103,2023-07-03,6.9
4,101,2023-07-04,7.2


material_id,material_name,material_type
101,Material A,Type X
102,Material B,Type Y
104,Material C,Type Z


experiment_id,material_id,experiment_date,experiment_results,material_id,material_name,material_type
1,101,2023-07-01,7.6,101,Material A,Type X
2,102,2023-07-02,8.3,102,Material B,Type Y
3,103,2023-07-03,6.9,null,null,null
4,101,2023-07-04,7.2,101,Material A,Type X


%md
## Temperature Reading Analysis

**Difficulty:** Easy

### Problem

Temperature Reading Analysis.

You are a data engineer at a thermodynamics lab. Temperature and pressure sensors log into separate tables, and the analysis team wants a combined metric per experiment; some experiments only have a reading in one of the two tables, and those cannot be scored and must be left out. Write a query that joins `td_temperature` to `td_pressure` on `experimentid`, keeping only experiments that appear in both tables, and computes the product `temperature * pressure` for each matched experiment. Return the experiment ID aliased exactly as `ExperimentID` and the product aliased exactly as `Result` (the physical columns are lowercase, so quoted aliases are required). Sort the results by experiment ID in ascending order.

**Schema columns:** `td_pressure.experimentid`, `td_pressure.pressure`, `td_temperature.experimentid`, `td_temperature.temperature`

**Output columns:** `ExperimentID`, `Result`

Order the result by `ExperimentID ASC`.

### Examples

#### Example 1

**Input:**

**td_pressure:**

| experimentid | pressure |
|-------------:|---------:|
| 1 | 1 |
| 3 | 2 |

**td_temperature:**

| experimentid | temperature |
|-------------:|------------:|
| 1 | 273.15 |
| 2 | 293.15 |
| 3 | 313.15 |

**Output:**

| ExperimentID | Result |
|-------------:|-------:|
| 1 | 273.15 |
| 3 | 626.3 |

**Explanation:** Experiment 1 appears in both tables, so its Result is 273.15 × 1 = 273.15; experiment 3 gives 313.15 × 2 = 626.3. Experiment 2 has a temperature reading but no pressure reading, so it is excluded by the inner join.

### Constraints

- Inner join `td_temperature` to `td_pressure` on `experimentid`; experiments missing from either table are excluded.
- `Result` = `temperature` × `pressure`.
- Output columns must be aliased exactly `ExperimentID` and `Result`.
- Sort by experiment ID in ascending order.

In [0]:
td_pressure_data=[(1,1),(3,2)]
td_pressure_df=spark.createDataFrame(td_pressure_data,["experimentid","pressure"])
display(td_pressure_df)

td_temperature_data=[(1,273.15),(2,293.15),(3,313.15)]
td_temperature_df=spark.createDataFrame(td_temperature_data,["experimentid","temperature"])
display(td_temperature_df)

joined_df = td_pressure_df.alias("p").join(td_temperature_df.alias("t"), on=col("p.experimentid") == col("t.experimentid"), how="inner")

output_df = (
joined_df.withColumn("Result", (col("p.pressure") * col("t.temperature")))
.select(col("p.experimentid").alias("ExperimentId"), col("Result"))
.orderBy("ExperimentId")
)

display(output_df)





experimentid,pressure
1,1
3,2


experimentid,temperature
1,273.15
2,293.15
3,313.15


ExperimentId,Result
1,273.15
3,626.3


## Que 03: List Comprehension and Cross Join

**Difficulty:** Easy

### Problem

A pricing team wants to see how every promotion would affect every product, so they can compare deal scenarios side by side. Pair each product with each promotion and report the resulting sale price.

For every product-promotion pairing, return one row with the product's id and name, the promotion's id and name, the product's `base_price`, and a `discounted_price` equal to the base price reduced by the promotion's discount percentage:

`base_price * (1 - discount_pct / 100)`

Round the result to 2 decimal places.

**Schema columns:** `products.product_id`, `products.product_name`, `products.category`, `products.base_price`, `promotions.promo_id`, `promotions.promo_name`, `promotions.discount_pct`

**Output columns:** `product_id`, `product_name`, `promo_id`, `promo_name`, `base_price`, `discounted_price`

Order the result by `product_id`, then by `promo_id`.

### Examples

#### Example 1

**Input:**

**products:**

| product_id | product_name | category | base_price |
|------------|--------------|-----------|-----------:|
| P001 | Laptop | Electronics | 999.99 |
| P002 | Mouse | Electronics | 25 |
| P005 | Desk Chair | Furniture | 199.99 |

**promotions:**

| promo_id | promo_name | discount_pct |
|----------|-------------|-------------:|
| PR001 | Summer Sale | 10 |
| PR002 | Flash Deal | 20 |

**Output:**

| product_id | product_name | promo_id | promo_name | base_price | discounted_price |
|------------|--------------|----------|------------|-----------:|-----------------:|
| P001 | Laptop | PR001 | Summer Sale | 999.99 | 899.99 |
| P001 | Laptop | PR002 | Flash Deal | 999.99 | 799.99 |
| P002 | Mouse | PR001 | Summer Sale | 25 | 22.50 |
| P002 | Mouse | PR002 | Flash Deal | 25 | 20.00 |
| P005 | Desk Chair | PR001 | Summer Sale | 199.99 | 179.99 |
| P005 | Desk Chair | PR002 | Flash Deal | 199.99 | 159.99 |

**Explanation:** With 3 products and 2 promotions, every product appears paired with both promotions, giving 6 rows. For the Laptop under Flash Deal, the 20% discount turns its 999.99 base price into `999.99 * (1 - 20/100) = 799.99`.

### Constraints

- Every product is paired with every promotion.
- `discounted_price` is rounded to 2 decimal places.
- A 0% discount leaves `base_price` unchanged; a 100% discount yields `0.00`.
- Results are ordered by `product_id`, then `promo_id`.
- Return results matching the expected output schema and order.

In [0]:
products_data=[("P001","Laptop","Electronics",999.99),("P002","Mouse","Electronics",25.0),("P005","Desk Chair","Furniture",199.99)]
products_df=spark.createDataFrame(products_data,["product_id","product_name","category","base_price"])
display(products_df)

promotions_data=[("PR001","Summer Sale",10),("PR002","Flash Deal",20)]
promotions_df=spark.createDataFrame(promotions_data,["promo_id","promo_name","discount_pct"])
display(promotions_df)

# product_id	product_name	promo_id	promo_name	base_price	discounted_price

# base_price * (1 - discount_pct / 100)

joined_df = products_df.alias("pd").join(promotions_df.alias("pr"), how="cross")

output_df = (
joined_df.
withColumn("discounted_price", round(( col("pd.base_price") * (1 - col("pr.discount_pct") / 100)  ) , 2))
.select("pd.product_id", "pd.product_name", "pr.promo_id", "pr.promo_name", "pd.base_price", "discounted_price")
.orderBy("pd.product_id", "pr.promo_id")
)

display(output_df)



product_id,product_name,category,base_price
P001,Laptop,Electronics,999.99
P002,Mouse,Electronics,25.0
P005,Desk Chair,Furniture,199.99


promo_id,promo_name,discount_pct
PR001,Summer Sale,10
PR002,Flash Deal,20


product_id,product_name,promo_id,promo_name,base_price,discounted_price
P001,Laptop,PR001,Summer Sale,999.99,899.99
P001,Laptop,PR002,Flash Deal,999.99,799.99
P002,Mouse,PR001,Summer Sale,25.0,22.5
P002,Mouse,PR002,Flash Deal,25.0,20.0
P005,Desk Chair,PR001,Summer Sale,199.99,179.99
P005,Desk Chair,PR002,Flash Deal,199.99,159.99


## Que 04: Rising Temperature (Day-over-Day)

**Difficulty:** Easy

### Problem

You are given a weather table with one temperature reading per calendar day. Find every reading whose temperature is strictly higher than the temperature recorded on the immediately preceding calendar day (exactly one day earlier). A reading whose previous calendar day is missing from the data cannot qualify.

Return the `id` of each qualifying reading.

**Schema columns:** `weather.id`, `weather.record_date`, `weather.temperature`

**Output columns:** `id`

Order the result by `id` ascending.

### Examples

#### Example 1

**Input:**

**weather:**

| id | record_date | temperature |
|---:|-------------|------------:|
| 1 | 2020-01-01 | 10 |
| 2 | 2020-01-02 | 25 |
| 3 | 2020-01-03 | 20 |
| 4 | 2020-01-04 | 30 |
| 5 | 2020-01-06 | 35 |
| 6 | 2020-01-07 | 40 |

**Output:**

| id |
|---:|
| 2 |
| 4 |
| 6 |

**Explanation:** Reading 2 on 2020-01-02 (25) is higher than reading 1 on 2020-01-01 (10), so it qualifies. Reading 4 (30) is higher than the previous day's 20, and reading 6 (40) is higher than the previous day's 35. Reading 3 is excluded because 20 is not greater than 25, and reading 5 is excluded because there is no record for 2020-01-05.

### Constraints

- Compare only readings whose dates are exactly one calendar day apart; gaps in the data disqualify a reading.
- The comparison is strict (greater than, not greater-or-equal).
- Each date appears at most once.
- Return results matching the expected output schema and order.

In [0]:
weather_data=[(1,"2020-01-01",10),(2,"2020-01-02",25),(3,"2020-01-03",20),(4,"2020-01-04",30),(5,"2020-01-06",35),(6,"2020-01-07",40)]
weather_df=spark.createDataFrame(weather_data,["id","record_date","temperature"])
display(weather_df)

window_spec = Window.orderBy(col("record_date"))

prev_data_df = (
weather_df.
withColumn("prev_temperature", lag("temperature").over(window_spec)).
withColumn("record_date", col("record_date").cast("date")).
withColumn("prev_record_date", lag("record_date").over(window_spec)).
withColumn("actual_prev_date", col("record_date").cast("date") - 1)
)


output_df = prev_data_df.filter(
    (col("actual_prev_date") == col("prev_record_date")) & 
    (col("temperature") > col("prev_temperature"))
).select("id")

display(output_df)



id,record_date,temperature
1,2020-01-01,10
2,2020-01-02,25
3,2020-01-03,20
4,2020-01-04,30
5,2020-01-06,35
6,2020-01-07,40


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id
2
4
6


## Que 05: Pages With No Likes

**Difficulty:** Easy

### Problem

Facebook wants to find pages that have never received a single like, so the content team can flag them for promotional attention. A page appears in the `page_likes` table once for every like it receives, and a page with no likes has no rows there at all.

Return the `page_id` of every page that has zero likes.

**Schema columns:** `page_likes.user_id`, `page_likes.page_id`, `page_likes.liked_date`, `pages.page_id`, `pages.page_name`

**Output columns:** `page_id`

Order the result by `page_id` ascending.

### Examples

#### Example 1

**Input:**

**page_likes:**

| user_id | page_id | liked_date |
|--------:|--------:|---------------------|
| 101 | 1 | 2024-01-15 10:30:00 |
| 102 | 1 | 2024-01-16 14:22:00 |
| 106 | 3 | 2024-01-10 08:20:00 |

**pages:**

| page_id | page_name |
|--------:|-----------------------|
| 1 | Meta Careers |
| 2 | Facebook Developers |
| 3 | Mark Zuckerberg |
| 4 | WhatsApp Official |
| 5 | Instagram Business |

**Output:**

| page_id |
|--------:|
| 2 |
| 4 |
| 5 |

**Explanation:** Pages 1 and 3 each appear in `page_likes`, so they are excluded. Pages 2, 4, and 5 have no rows in `page_likes`, so they are returned in ascending order of `page_id`.

### Constraints

- Return only `page_id` in the output (not `page_name`).
- A page qualifies only if it has exactly zero likes.
- Sort the results by `page_id` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
page_likes_data=[(101,1,"2024-01-15 10:30:00"),(102,1,"2024-01-16 14:22:00"),(106,3,"2024-01-10 08:20:00")]
page_likes_df=spark.createDataFrame(page_likes_data,["user_id","page_id","liked_date"])
display(page_likes_df)

pages_data=[(1,"Meta Careers"),(2,"Facebook Developers"),(3,"Mark Zuckerberg"),(4,"WhatsApp Official"),(5,"Instagram Business")]
pages_df=spark.createDataFrame(pages_data,["page_id","page_name"])
display(pages_df)

output_df = pages_df.alias("pg").join(page_likes_df.alias("pl"), on=col("pg.page_id") == col("pl.page_id"), how="left").filter(col("pl.page_id").isNull()).select("pg.page_id").orderBy("pg.page_id")

display(output_df)


user_id,page_id,liked_date
101,1,2024-01-15 10:30:00
102,1,2024-01-16 14:22:00
106,3,2024-01-10 08:20:00


page_id,page_name
1,Meta Careers
2,Facebook Developers
3,Mark Zuckerberg
4,WhatsApp Official
5,Instagram Business


page_id
2
4
5


## Que 06: List African Cities

**Difficulty:** Easy

### Problem

A geography report needs the unique names of cities whose countries are in Africa. Return one row per city name, even if duplicate city records exist.

**Schema columns:** `city.id`, `city.name`, `city.countrycode`, `city.population`, `country.code`, `country.name`, `country.continent`

**Output columns:** `city_name`

Order the result by `city_name` ascending.

### Examples

#### Example 1

**Input:**

**city:**

| id | name | countrycode | population |
|---:|------|-------------|-----------:|
| 6 | Cairo | EGY | 20484965 |
| 20 | Cairo | EGY | 20484965 |
| 13 | Lagos | NGA | 13463934 |

**country:**

| code | name | continent |
|------|-------|-----------|
| EGY | Egypt | Africa |
| NGA | Nigeria | Africa |

**Output:**

| city_name |
|-----------|
| Cairo |
| Lagos |

**Explanation:** Both Cairo records collapse into one unique city name because only distinct city names are required. Lagos is also included since Nigeria is in Africa. The results are sorted alphabetically by `city_name`.

### Constraints

- Return unique city names only.
- Include cities whose country's `continent` is exactly `Africa`.
- Sort the results by `city_name` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
city_data=[(6,"Cairo","EGY",20484965),(20,"Cairo","EGY",20484965),(13,"Lagos","NGA",13463934)]
city_df=spark.createDataFrame(city_data,["id","name","countrycode","population"])
display(city_df)

country_data=[("EGY","Egypt","Africa"),("NGA","Nigeria","Africa")]
country_df=spark.createDataFrame(country_data,["code","name","continent"])
display(country_df)


joined_df = city_df.alias("ct").join(country_df.alias("cn"), on=((col("ct.countrycode") == col("cn.code")) & (col("cn.continent") == "Africa")), how="inner")

output_df = joined_df.select("ct.name").distinct().orderBy("ct.name")

display(output_df)



id,name,countrycode,population
6,Cairo,EGY,20484965
20,Cairo,EGY,20484965
13,Lagos,NGA,13463934


code,name,continent
EGY,Egypt,Africa
NGA,Nigeria,Africa


name
Cairo
Lagos


%md
## List the Products Ordered in a Period

**Difficulty:** Easy

### Problem

A retailer wants products with at least 100 units ordered during February 2020. Sum units for orders on or after `2020-02-01` and before `2020-03-01`, and return only products whose period total is at least 100.

**Schema columns:** `orders.order_id`, `orders.product_id`, `orders.order_date`, `orders.unit`, `products.product_id`, `products.product_name`, `products.category`

**Output columns:** `product_name`, `unit`

Order the result by `unit` descending, then `product_name` ascending.

### Examples

#### Example 1

**Input:**

**orders:**

| order_id | product_id | order_date | unit |
|---------:|-----------:|------------|-----:|
| 1 | 1 | 2020-02-01 | 45 |
| 2 | 1 | 2020-02-05 | 30 |
| 3 | 1 | 2020-02-10 | 35 |
| 4 | 2 | 2020-02-03 | 45 |
| 5 | 2 | 2020-02-12 | 30 |
| 17 | 2 | 2020-02-28 | 20 |

**products:**

| product_id | product_name | category |
|-----------:|--------------|-----------|
| 1 | Laptop | Electronics |
| 2 | Mouse | Electronics |

**Output:**

| product_name | unit |
|--------------|-----:|
| Laptop | 110 |

**Explanation:** Laptop has a total of 110 units ordered during February 2020, so it qualifies. Mouse totals only 95 units during the same period and is excluded.

### Constraints

- Consider only orders from `2020-02-01` (inclusive) to `2020-03-01` (exclusive).
- Include products whose total February units are greater than or equal to 100.
- Sort the results by `unit` descending, then `product_name` ascending.
- Return results matching the expected output schema and order.

In [0]:
orders_data=[(1,1,"2020-02-01",45),(2,1,"2020-02-05",30),(3,1,"2020-02-10",35),(4,2,"2020-02-03",45),(5,2,"2020-02-12",30),(17,2,"2020-02-28",20),(18,2,"2020-03-28",20)]
orders_df=spark.createDataFrame(orders_data,["order_id","product_id","order_date","unit"])
display(orders_df)

products_data=[(1,"Laptop","Electronics"),(2,"Mouse","Electronics")]
products_df=spark.createDataFrame(products_data,["product_id","product_name","category"])
display(products_df)

feb_orders_df = orders_df.filter(col("order_date").between("2020-02-01", "2020-02-29"))

grouped_df = (
feb_orders_df.groupBy("product_id").agg(
    sum("unit").alias("total_unit")
)
.filter(col("total_unit") > 100)
)

joined_df = products_df.alias("pd").join(grouped_df.alias("gd"), on="product_id", how="inner")

output_df = joined_df.select("product_name", "total_unit").orderBy(col("gd.total_unit").desc(), col("pd.product_name"))

display(output_df)


order_id,product_id,order_date,unit
1,1,2020-02-01,45
2,1,2020-02-05,30
3,1,2020-02-10,35
4,2,2020-02-03,45
5,2,2020-02-12,30
17,2,2020-02-28,20
18,2,2020-03-28,20


product_id,product_name,category
1,Laptop,Electronics
2,Mouse,Electronics


product_name,total_unit
Laptop,110


%md
## Average Time of Process per Machine

**Difficulty:** Easy

### Problem

A factory records a start and end timestamp for each process on each machine. For every machine, return the average elapsed time between matching start and end events.

**Schema columns:** `activity.machine_id`, `activity.process_id`, `activity.activity_type`, `activity.timestamp`

**Output columns:** `machine_id`, `processing_time`

Order the result by `machine_id` ascending.

### Examples

#### Example 1

**Input:**

**activity:**

| machine_id | process_id | activity_type | timestamp |
|-----------:|-----------:|---------------|----------:|
| 1 | 101 | start | 1000 |
| 1 | 101 | end | 1500 |
| 1 | 102 | start | 2000 |
| 1 | 102 | end | 2800 |
| 1 | 103 | start | 3000 |
| 1 | 103 | end | 3300 |

**Output:**

| machine_id | processing_time |
|-----------:|----------------:|
| 1 | 533.333 |

**Explanation:** For machine 1, the three process durations are 500, 800, and 300. Their average is `(500 + 800 + 300) / 3 = 533.333`, rounded to three decimal places.

### Constraints

- Match each `start` event with its corresponding `end` event using both `machine_id` and `process_id`.
- Calculate the average processing time for each machine.
- Round `processing_time` to 3 decimal places.
- Sort the results by `machine_id` in ascending order.

In [0]:
activity_data=[(1,101,"start",1000),(1,101,"end",1500),(1,102,"start",2000),(1,102,"end",2800),(1,103,"start",3000),(1,103,"end",3300)]
activity_df=spark.createDataFrame(activity_data,["machine_id","process_id","activity_type","timestamp"])
display(activity_df)

joined_df = activity_df.alias("a1").join(activity_df.alias("a2"), 
on=(col("a1.machine_id") == col("a2.machine_id")) & (col("a1.process_id") == col("a2.process_id")))


start_end_df = (
joined_df.
select(col("a1.machine_id"), col("a1.process_id"), col("a1.timestamp").alias("start_timestamp"), col("a2.timestamp").alias("end_timestamp")).
withColumn("duration", col("end_timestamp") - col("start_timestamp")).
filter((col("a1.activity_type") == "start") & (col("a2.activity_type") == "end"))
)

output_df = (
start_end_df.groupBy("machine_id").agg(
    round(avg("duration"), 3).alias("processing_time")
).
orderBy("machine_id"))

display(output_df)


machine_id,process_id,activity_type,timestamp
1,101,start,1000
1,101,end,1500
1,102,start,2000
1,102,end,2800
1,103,start,3000
1,103,end,3300


machine_id,processing_time
1,533.333


%md
## The Number of Employees Which Report to Each Employee

**Difficulty:** Easy

### Problem

An HR team is preparing a manager overview. For every employee who manages at least one direct report, return the manager's id and name, the number of employees who report directly to them, and the rounded average age of those direct reports.

**Schema columns:** `employees.employee_id`, `employees.manager_id`, `employees.name`, `employees.age`

**Output columns:** `employee_id`, `name`, `reports_count`, `average_age`

Order the result by `employee_id` ascending.

### Examples

#### Example 1

**Input:**

**employees:**

| employee_id | manager_id | name | age |
|------------:|-----------:|------|----:|
| 101 | NULL | CEO | 55 |
| 102 | 101 | Alice | 42 |
| 103 | 101 | Bob | 46 |
| 104 | 102 | Charlie | 30 |
| 105 | 102 | Diana | 36 |

**Output:**

| employee_id | name | reports_count | average_age |
|------------:|------|--------------:|------------:|
| 101 | CEO | 2 | 44 |
| 102 | Alice | 2 | 33 |

**Explanation:** The CEO directly manages Alice and Bob, whose average age is 44. Alice directly manages Charlie and Diana, whose average age is 33. Only employees with at least one direct report are included.

### Constraints

- Count only direct reports, not indirect reports.
- Return only employees who manage at least one direct report.
- Round the average age to the nearest integer.
- Sort the results by `employee_id` in ascending order.

In [0]:
employees_data=[(101,None,"CEO",55),(102,101,"Alice",42),(103,101,"Bob",47),(104,102,"Charlie",30),(105,102,"Diana",36)]

employees_df=spark.createDataFrame(employees_data,["employee_id","manager_id","name","age"])

display(employees_df)


joined_df = employees_df.alias("m").join(employees_df.alias("e"), on=col("m.employee_id") == col("e.manager_id"), how="inner")

grouped_df = joined_df.groupBy("m.employee_id", "m.name").agg(
    count("e.employee_id").alias("reports_count"),
    round(avg(col("e.age"))).alias("average_age")
).filter(col("reports_count") >= 1).orderBy("m.employee_id")

display(grouped_df)



employee_id,manager_id,name,age
101,null,CEO,55
102,101,Alice,42
103,101,Bob,47
104,102,Charlie,30
105,102,Diana,36


employee_id,name,reports_count,average_age
101,CEO,2,45.0
102,Alice,2,33.0


%md
## Customers Who Bought Only a Single Item

**Difficulty:** Easy

### Problem

Amazon wants to find customers who are true one-time buyers: they placed exactly one order, and that single order contained exactly one distinct product. These customers are prime targets for follow-up and loyalty campaigns.

For each such customer, return their `customer_id`, `customer_name`, the `product_name` they bought, the `order_date` of that order, and `total_spend`, which is calculated as `quantity × price` rounded to 2 decimal places. If one order contains multiple line items for the same product, treat it as a single distinct product.

**Schema columns:** `customers.customer_id`, `customers.customer_name`, `order_items.order_item_id`, `order_items.order_id`, `order_items.product_id`, `order_items.product_name`, `order_items.quantity`, `order_items.price`, `orders.order_id`, `orders.customer_id`, `orders.order_date`

**Output columns:** `customer_id`, `customer_name`, `product_name`, `order_date`, `total_spend`

Order the result by `customer_id` ascending.

### Examples

#### Example 1

**Input:**

**customers:**

| customer_id | customer_name |
|------------:|---------------|
| 101 | Alice Johnson |
| 102 | Bob Smith |
| 103 | Charlie Brown |
| 106 | Frank Miller |

**orders:**

| order_id | customer_id | order_date |
|---------:|------------:|------------|
| 1001 | 101 | 2024-01-10 |
| 1010 | 101 | 2024-02-15 |
| 1002 | 102 | 2024-01-12 |
| 1003 | 103 | 2024-01-15 |
| 1006 | 106 | 2024-01-25 |

**order_items:**

| order_item_id | order_id | product_id | product_name | quantity | price |
|--------------:|---------:|-----------:|--------------|---------:|------:|
| 1 | 1001 | 501 | Laptop | 1 | 1500 |
| 11 | 1010 | 510 | Mouse Pad | 1 | 20 |
| 2 | 1002 | 502 | Mouse | 2 | 25 |
| 3 | 1003 | 503 | Keyboard | 1 | 80 |
| 6 | 1006 | 506 | Headphones | 1 | 150 |
| 7 | 1006 | 507 | Charger | 1 | 50 |

**Output:**

| customer_id | customer_name | product_name | order_date | total_spend |
|------------:|---------------|--------------|------------|------------:|
| 102 | Bob Smith | Mouse | 2024-01-12 | 50.00 |
| 103 | Charlie Brown | Keyboard | 2024-01-15 | 80.00 |

**Explanation:** Bob Smith qualifies because he placed exactly one order and that order contains only one distinct product. Charlie Brown also qualifies for the same reason. Alice Johnson is excluded because she placed two orders, while Frank Miller is excluded because his single order contains two different products.

### Constraints

- Include only customers with exactly one order.
- That order must contain exactly one distinct product. Multiple line items of the same product count as one product.
- Calculate `total_spend` as `quantity × price`, rounded to 2 decimal places.
- Sort the results by `customer_id` in ascending order.

In [0]:
from pyspark.sql.functions import *

customers_data=[(101,"Alice Johnson"),(102,"Bob Smith"),(103,"Charlie Brown"),(106,"Frank Miller")]
customers_df=spark.createDataFrame(customers_data,["customer_id","customer_name"])

orders_data=[(1001,101,"2024-01-10"),(1010,101,"2024-02-15"),(1002,102,"2024-01-12"),(1003,103,"2024-01-15"),(1006,106,"2024-01-25")]
orders_df=spark.createDataFrame(orders_data,["order_id","customer_id","order_date"])

order_items_data=[
    (1,1001,501,"Laptop",1,1500.0),
    (11,1010,510,"Mouse Pad",1,20.0),
    (2,1002,502,"Mouse",2,25.0),
    (3,1003,503,"Keyboard",1,80.0),
    (6,1006,506,"Headphones",1,150.0),
    (7,1006,507,"Charger",1,50.0)
]
order_items_df=spark.createDataFrame(
    order_items_data,
    ["order_item_id","order_id","product_id","product_name","quantity","price"]
)


# Customers with exactly one order
customer_grouped_df = (
    orders_df
    .groupBy("customer_id")
    .agg(count("order_id").alias("order_count"))
    .filter(col("order_count") == 1)
)

exactly_one_order_df = (
    customer_grouped_df
    .join(orders_df, on="customer_id", how="inner")
)


# Orders having exactly one DISTINCT product
product_grouped_df = (
    order_items_df
    .groupBy("order_id")
    .agg(
        countDistinct("product_id").alias("product_count"),
        first("product_name").alias("product_name"),
        round(sum(col("quantity") * col("price")), 2).alias("total_spend")
    )
    .filter(col("product_count") == 1)
)


# Final Output
output_df = (
    exactly_one_order_df.alias("o")
    .join(product_grouped_df.alias("p"), on="order_id", how="inner")
    .join(customers_df.alias("c"), on="customer_id", how="inner")
    .select(
        "customer_id",
        "customer_name",
        "product_name",
        "order_date",
        "total_spend"
    )
    .orderBy("customer_id")
)

display(output_df)

customer_id,customer_name,product_name,order_date,total_spend
102,Bob Smith,Mouse,2024-01-12,50.0
103,Charlie Brown,Keyboard,2024-01-15,80.0


%md
## Customers Who Never Order

**Difficulty:** Easy

### Problem

A retail team keeps a list of registered customers and a log of the orders they place. Find every customer who has never placed a single order.

Return one row per such customer with their `customer_name`.

**Schema columns:** `customers.id`, `customers.name`, `orders.id`, `orders.customer_id`, `orders.order_date`, `orders.amount`

**Output columns:** `customer_name`

Order the result by `customer_name` ascending.

### Examples

#### Example 1

**Input:**

**customers:**

| id | name |
|---:|------|
| 1 | Alice |
| 2 | Bob |
| 3 | Charlie |
| 4 | David |
| 5 | Eve |

**orders:**

| id | customer_id | order_date | amount |
|---:|------------:|------------|-------:|
| 1 | 1 | 2023-01-01 | 100 |
| 2 | 3 | 2023-01-05 | 200 |
| 3 | 1 | 2023-02-01 | 150 |

**Output:**

| customer_name |
|---------------|
| Bob |
| David |
| Eve |

**Explanation:** Alice and Charlie have placed at least one order, so they are excluded. Bob, David, and Eve have never placed an order, so they are returned in alphabetical order.

### Constraints

- A customer is considered to have never ordered only if no order references their id.
- Match customers and orders using `orders.customer_id = customers.id`.
- Sort the results by `customer_name` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
customers_data=[(1,"Alice"),(2,"Bob"),(3,"Charlie"),(4,"David"),(5,"Eve")]
customers_df=spark.createDataFrame(customers_data,["id","name"])
display(customers_df)

orders_data=[(1,1,"2023-01-01",100),(2,3,"2023-01-05",200),(3,1,"2023-02-01",150)]
orders_df=spark.createDataFrame(orders_data,["id","customer_id","order_date","amount"])
display(orders_df)

joined_df = customers_df.alias("c").join(orders_df.alias("o"), on=col("c.id") == col("o.customer_id"), how="left")

output_df = joined_df.select(col("c.name").alias("customer_name")).filter("o.customer_id is NULL").orderBy("customer_name")

display(output_df)



id,name
1,Alice
2,Bob
3,Charlie
4,David
5,Eve


id,customer_id,order_date,amount
1,1,2023-01-01,100
2,3,2023-01-05,200
3,1,2023-02-01,150


customer_name
Bob
David
Eve


%md
## Students With Invalid Departments

**Difficulty:** Easy

### Problem

A school wants students whose department assignment is missing or does not identify a valid department. Return the affected student `id` and `name`.

**Schema columns:** `departments.id`, `departments.name`, `students.id`, `students.name`, `students.department_id`

**Output columns:** `id`, `name`

Order the result by `id` ascending.

### Examples

#### Example 1

**Input:**

**departments:**

| id | name |
|---:|-------------|
| 1 | Engineering |
| 2 | Sales |

**students:**

| id | name | department_id |
|---:|------|--------------:|
| 10 | Ava | 1 |
| 11 | Ben | 9 |
| 12 | Cara | NULL |

**Output:**

| id | name |
|---:|------|
| 11 | Ben |
| 12 | Cara |

**Explanation:** Ben references department 9, which does not exist. Cara has a NULL department_id. Ava belongs to a valid department and is excluded.

### Constraints

- Department identifiers are unique.
- A NULL or unmatched `department_id` is considered invalid.
- Sort the results by `id` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
departments_data=[(1,"Engineering"),(2,"Sales")]
departments_df=spark.createDataFrame(departments_data,["id","name"])
display(departments_df)

students_data=[(10,"Ava",1),(11,"Ben",9),(12,"Cara",None)]
students_df=spark.createDataFrame(students_data,["id","name","department_id"])
display(students_df)

joined_df = students_df.alias("s").join(departments_df.alias("d"), on=col("s.department_id") == col("d.id"), how="left")

output_df = joined_df.select("s.id", "s.name").filter("d.id is NULL").orderBy("s.id")

display(output_df)



id,name
1,Engineering
2,Sales


id,name,department_id
10,Ava,1
11,Ben,9
12,Cara,null


id,name
11,Ben
12,Cara


%md
## Employees Not in Department

**Difficulty:** Easy

### Problem

A data integrity audit needs to identify orphaned employee records—employees whose `department_id` references a non-existent department. This indicates missing or deleted department records.

**Schema columns:** `departments.department_id`, `departments.department_name`, `employees.employee_id`, `employees.name`, `employees.department_id`

**Output columns:** `employee_id`, `name`, `department_id`

Order the result by `employee_id` ascending.

### Examples

#### Example 1

**Input:**

**departments:**

| department_id | department_name |
|--------------:|-----------------|
| 1 | Engineering |
| 2 | Sales |
| 3 | HR |

**employees:**

| employee_id | name | department_id |
|------------:|------|--------------:|
| 1 | Alice | 1 |
| 2 | Bob | 2 |
| 3 | Charlie | 999 |
| 4 | David | 1 |
| 5 | Eve | 888 |

**Output:**

| employee_id | name | department_id |
|------------:|------|--------------:|
| 3 | Charlie | 999 |
| 5 | Eve | 888 |

**Explanation:** Charlie and Eve reference department IDs that do not exist in the departments table, so they are returned. Alice, Bob, and David belong to valid departments and are excluded.

### Constraints

- Return only employees whose `department_id` does not exist in the departments table.
- Include the invalid `department_id` in the output.
- Sort the results by `employee_id` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
departments_data=[(1,"Engineering"),(2,"Sales"),(3,"HR")]
departments_df=spark.createDataFrame(departments_data,["department_id","department_name"])
display(departments_df)

employees_data=[(1,"Alice",1),(2,"Bob",2),(3,"Charlie",999),(4,"David",1),(5,"Eve",888)]
employees_df=spark.createDataFrame(employees_data,["employee_id","name","department_id"])
display(employees_df)


joined_df = employees_df.alias("e").join(departments_df.alias("d"), on=col("e.department_id") == col("d.department_id"), how="left")

output_df = joined_df.select("e.*").filter("d.department_id is NULL").orderBy("e.employee_id")

display(output_df)


department_id,department_name
1,Engineering
2,Sales
3,HR


employee_id,name,department_id
1,Alice,1
2,Bob,2
3,Charlie,999
4,David,1
5,Eve,888


employee_id,name,department_id
3,Charlie,999
5,Eve,888


%md
## Employees Without Managers

**Difficulty:** Easy

### Problem

A company keeps one row per employee with their salary and the id of the manager they report to. Some listed managers have since left, so their id no longer matches any current employee. Find every employee who earns less than 30000 and whose manager is not among the current employees, including employees who have no manager on record.

Return each qualifying `employee_id`.

**Schema columns:** `mmf_employees_sample.employee_id`, `mmf_employees_sample.emp_name`, `mmf_employees_sample.manager_id`, `mmf_employees_sample.salary`

**Output columns:** `employee_id`

Order the result by `employee_id` ascending.

### Examples

#### Example 1

**Input:**

**mmf_employees_sample:**

| employee_id | emp_name | manager_id | salary |
|------------:|----------|-----------:|-------:|
| 1 | Kalel | 11 | 21241 |
| 3 | Mila | 9 | 60301 |
| 9 | Mikaela | NULL | 50937 |
| 11 | Joziah | 6 | 28485 |
| 12 | Antonella | NULL | 19500 |

**Output:**

| employee_id |
|------------:|
| 11 |
| 12 |

**Explanation:** Joziah earns less than 30000 and reports to manager 6, whose id does not exist in the employee table. Antonella also earns less than 30000 and has no manager assigned. Kalel is excluded because his manager exists, while Mila and Mikaela are excluded because their salaries are at least 30000.

### Constraints

- Consider only employees with a salary below 30000.
- An employee qualifies if their `manager_id` does not exist as an `employee_id` or if `manager_id` is NULL.
- Sort the results by `employee_id` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
mmf_employees_sample_data=[(1,"Kalel",11,21241),(3,"Mila",9,60301),(9,"Mikaela",None,50937),(11,"Joziah",6,28485),(12,"Antonella",None,19500)]
mmf_employees_sample_df=spark.createDataFrame(mmf_employees_sample_data,["employee_id","emp_name","manager_id","salary"])
display(mmf_employees_sample_df)

joined_df = mmf_employees_sample_df.alias("e").join(mmf_employees_sample_df.alias("m"), on=col("e.manager_id") == col("m.employee_id"), how="left")

output_df = joined_df.select("e.employee_id").where((col("m.employee_id").isNull()) & (col("e.salary") < 30000)).orderBy("e.employee_id")

display(output_df)



employee_id,emp_name,manager_id,salary
1,Kalel,11,21241
3,Mila,9,60301
9,Mikaela,null,50937
11,Joziah,6,28485
12,Antonella,null,19500


employee_id
11
12


%md
## Employee ID Cross-Reference

**Difficulty:** Easy

### Problem

Each employee has a name, and some (but not all) employees have been assigned a unique ID. For every employee, return their assigned `unique_id` alongside their `name`. When an employee has no assigned unique ID, return `NULL` for that value instead.

**Schema columns:** `euim_employee_uni_sample.id`, `euim_employee_uni_sample.unique_id`, `euim_employees_sample.id`, `euim_employees_sample.name`

**Output columns:** `unique_id`, `name`

The result may be returned in any order.

### Examples

#### Example 1

**Input:**

**euim_employee_uni_sample:**

| id | unique_id |
|---:|----------:|
| 3 | 1 |
| 11 | 2 |
| 90 | 3 |

**euim_employees_sample:**

| id | name |
|---:|----------|
| 1 | Alice |
| 7 | Bob |
| 11 | Meir |
| 90 | Winston |
| 3 | Jonathan |

**Output:**

| unique_id | name |
|----------:|----------|
| NULL | Alice |
| 1 | Jonathan |
| NULL | Bob |
| 2 | Meir |
| 3 | Winston |

**Explanation:** Jonathan has employee ID 3, which maps to unique ID 1. Meir and Winston also have assigned unique IDs. Alice and Bob have no matching record in the employee unique ID table, so their `unique_id` is returned as `NULL`.

### Constraints

- Every employee must appear exactly once in the result.
- Return `NULL` for employees without an assigned unique ID.
- Match records using the employee `id`.
- Return results matching the expected output schema.

In [0]:
euim_employee_uni_sample_data=[(3,1),(11,2),(90,3)]
euim_employee_uni_sample_df=spark.createDataFrame(euim_employee_uni_sample_data,["id","unique_id"])
display(euim_employee_uni_sample_df)

euim_employees_sample_data=[(1,"Alice"),(7,"Bob"),(11,"Meir"),(90,"Winston"),(3,"Jonathan")]
euim_employees_sample_df=spark.createDataFrame(euim_employees_sample_data,["id","name"])
display(euim_employees_sample_df)


joined_df = euim_employees_sample_df.alias("e").join(euim_employee_uni_sample_df.alias("u"), on=col("e.id") == col("u.id"), how="left")

output_df = joined_df.select("unique_id", "name")


display(output_df)




id,unique_id
3,1
11,2
90,3


id,name
1,Alice
7,Bob
11,Meir
90,Winston
3,Jonathan


unique_id,name
null,Alice
null,Bob
2,Meir
3,Winston
1,Jonathan


%md
## Enrich Events with Countries Using Broadcast Variables

**Difficulty:** Medium

### Problem

Real-time event data streams in with country codes, but you need to enrich each event with full country names and regions. The countries dimension is small and static, making it ideal for broadcast.

**Schema columns:** `countries.country_code`, `countries.country_name`, `countries.region`, `events.event_id`, `events.user_id`, `events.event_type`, `events.country_code`, `events.event_date`

**Output columns:** `event_id`, `user_id`, `event_type`, `country_name`, `region`, `event_date`

Order the result by `event_id` ascending.

### Examples

#### Example 1

**Input:**

**countries:**

| country_code | country_name | region |
|--------------|----------------|----------------|
| US | United States | North America |
| UK | United Kingdom | Europe |
| IN | India | Asia |
| CA | Canada | North America |

**events:**

| event_id | user_id | event_type | country_code | event_date |
|---------:|--------:|------------|--------------|------------|
| 1 | 62 | click | XX | 2024-01-08 |
| 2 | 2 | purchase | IN | 2024-01-04 |
| 3 | 88 | click | CA | 2024-01-04 |
| 4 | 57 | view | CA | 2024-01-10 |

**Output:**

| event_id | user_id | event_type | country_name | region | event_date |
|---------:|--------:|------------|--------------|----------------|------------|
| 1 | 62 | click | NULL | NULL | 2024-01-08 |
| 2 | 2 | purchase | India | Asia | 2024-01-04 |
| 3 | 88 | click | Canada | North America | 2024-01-04 |
| 4 | 57 | view | Canada | North America | 2024-01-10 |

**Explanation:** Each event is enriched with the matching country name and region using the country code. Events with unknown country codes retain the event details and return `NULL` for `country_name` and `region`.

### Constraints

- Use a LEFT JOIN to preserve all events.
- Unknown country codes should return `NULL` for `country_name` and `region`.
- Handle NULL and empty values gracefully.
- In PySpark, use `broadcast()` for the small countries table.

In [0]:
countries_data=[("US","United States","North America"),("UK","United Kingdom","Europe"),("IN","India","Asia"),("CA","Canada","North America")]
countries_df=spark.createDataFrame(countries_data,["country_code","country_name","region"])
display(countries_df)

events_data=[(1,62,"click","XX","2024-01-08"),(2,2,"purchase","IN","2024-01-04"),(3,88,"click","CA","2024-01-04"),(4,57,"view","CA","2024-01-10")]
events_df=spark.createDataFrame(events_data,["event_id","user_id","event_type","country_code","event_date"])
display(events_df)

# event_id	user_id	event_type	country_name	region	event_date

joined_df = (
events_df.alias("e")
.join(broadcast(countries_df).alias('c'), 
    on=col("e.country_code") == col("c.country_code"), how="left")
)


output_df = joined_df.select("e.event_id","e.user_id","e.event_type","c.country_name","c.region","e.event_date")

display(output_df)








country_code,country_name,region
US,United States,North America
UK,United Kingdom,Europe
IN,India,Asia
CA,Canada,North America


event_id,user_id,event_type,country_code,event_date
1,62,click,XX,2024-01-08
2,2,purchase,IN,2024-01-04
3,88,click,CA,2024-01-04
4,57,view,CA,2024-01-10


event_id,user_id,event_type,country_name,region,event_date
1,62,click,null,null,2024-01-08
2,2,purchase,India,Asia,2024-01-04
3,88,click,Canada,North America,2024-01-04
4,57,view,Canada,North America,2024-01-10


## F&B Product Sales Summary

**Difficulty:** Medium

### Problem

A food-and-beverage retailer needs one summary row for every product, including products with no sales. Report total quantity and revenue from sales, along with total stock across warehouses. Use zero when a product has no matching sales or inventory records.

**Schema columns:** `fnb_products.product_id`, `fnb_products.name`, `fnb_products.category`, `fnb_sales.sale_id`, `fnb_sales.product_id`, `fnb_sales.quantity`, `fnb_sales.revenue`, `fnb_inventory.product_id`, `fnb_inventory.stock`, `fnb_inventory.warehouse`

**Output columns:** `product_id`, `name`, `category`, `total_quantity`, `total_revenue`, `total_stock`

Order the result by `category` ascending, then `product_id` ascending.

### Examples

#### Example 1

**Input:**

**fnb_products:**

| product_id | name | category |
|-----------:|----------------|-----------|
| 1 | Orange Juice | Beverages |
| 2 | Green Tea | Beverages |
| 3 | Chocolate Bar | Snacks |
| 4 | Potato Chips | Snacks |
| 5 | Cookies | Snacks |

**fnb_sales:**

| sale_id | product_id | quantity | revenue |
|--------:|-----------:|---------:|--------:|
| 1 | 1 | 10 | 50 |
| 2 | 3 | 2 | 4 |
| 3 | 4 | 15 | 30 |

**fnb_inventory:**

| product_id | stock | warehouse |
|-----------:|------:|------------|
| 1 | 25 | Warehouse1 |
| 2 | 40 | Warehouse1 |
| 3 | 30 | Warehouse1 |

**Output:**

| product_id | name | category | total_quantity | total_revenue | total_stock |
|-----------:|----------------|-----------|---------------:|--------------:|------------:|
| 1 | Orange Juice | Beverages | 10 | 50 | 25 |
| 2 | Green Tea | Beverages | 0 | 0 | 40 |
| 3 | Chocolate Bar | Snacks | 2 | 4 | 30 |
| 4 | Potato Chips | Snacks | 15 | 30 | 0 |
| 5 | Cookies | Snacks | 0 | 0 | 0 |

**Explanation:**

- **Orange Juice** has both sales and inventory.
- **Green Tea** has inventory but no sales, so quantity and revenue are **0**.
- **Chocolate Bar** has both sales and inventory.
- **Potato Chips** has sales but no inventory, so stock is **0**.
- **Cookies** has neither sales nor inventory, so all aggregated values are **0**.

### Constraints

- Include every product from `fnb_products`.
- Replace missing sales or inventory totals with zero.
- Sort by `category`, then `product_id`, both in ascending order.
- Return results matching the expected output schema and order.

In [0]:
# Products
fnb_products_data = [(1, "Orange Juice", "Beverages"),(2, "Green Tea", "Beverages"),(3, "Chocolate Bar", "Snacks"),(4, "Potato Chips", "Snacks"),(5, "Cookies", "Snacks")]
fnb_products_df = spark.createDataFrame(fnb_products_data,["product_id", "name", "category"])
display(fnb_products_df)


# Sales
fnb_sales_data = [(1, 1, 10, 50),(2, 3, 2, 4),(3, 4, 15, 30)]
fnb_sales_df = spark.createDataFrame(fnb_sales_data,["sale_id", "product_id", "quantity", "revenue"])
display(fnb_sales_df)


# Inventory
fnb_inventory_data = [(1, 25, "Warehouse1"),(2, 40, "Warehouse1"),(3, 30, "Warehouse1")]
fnb_inventory_df = spark.createDataFrame(fnb_inventory_data,["product_id", "stock", "warehouse"])
display(fnb_inventory_df)

# product_id	name	category	total_quantity	total_revenue	total_stock
sales_df = (
fnb_sales_df.groupBy("product_id").agg(
    sum("quantity").alias("total_quantity"),
    sum("revenue").alias("total_revenue")
))

inventory_df = (
fnb_inventory_df.groupBy("product_id").agg(
    sum("stock").alias("total_stock")
))

result = (
fnb_products_df
    .join(sales_df, "product_id", "left")
    .join(inventory_df, "product_id", "left")
    .fillna(0)
    .orderBy("category", "product_id")
)

display(result)

product_id,name,category
1,Orange Juice,Beverages
2,Green Tea,Beverages
3,Chocolate Bar,Snacks
4,Potato Chips,Snacks
5,Cookies,Snacks


sale_id,product_id,quantity,revenue
1,1,10,50
2,3,2,4
3,4,15,30


product_id,stock,warehouse
1,25,Warehouse1
2,40,Warehouse1
3,30,Warehouse1


product_id,name,category,total_quantity,total_revenue,total_stock
1,Orange Juice,Beverages,10,50,25
2,Green Tea,Beverages,0,0,40
3,Chocolate Bar,Snacks,2,4,30
4,Potato Chips,Snacks,15,30,0
5,Cookies,Snacks,0,0,0


## Mortgage Rate Comparison

**Difficulty:** Medium

### Problem

You are working for a mortgage company that manages various mortgage types selected by multiple users. The company stores this data in two DataFrames:

- **MortgageDetails:** Contains details of each mortgage type.
- **UserMortgages:** Tracks which users are associated with each mortgage.

Each mortgage is uniquely identified by a `MortgageID`, and each user by a `UserID`.

Write a function `etl(MortgageDetails, UserMortgages)` that returns a DataFrame containing the rate of mortgage for each `MortgageType`.

**Schema columns:** `mirmortgagedetails.MortgageID`, `mirmortgagedetails.MortgageType`, `mirmortgagedetails.InterestRate`, `mirusermortgages.UserID`, `mirusermortgages.MortgageID`

**Output columns:** `MortgageType`, `RateOfMortgage`

The result may be returned in any order.

### Examples

#### Example 1

**Input:**

**mirmortgagedetails:**

| MortgageID | MortgageType | InterestRate |
|------------|--------------|-------------:|
| M1 | Fixed | 4.5 |
| M2 | Variable | 3.2 |
| M3 | Adjustable | 2.8 |

**mirusermortgages:**

| UserID | MortgageID |
|--------|------------|
| U1 | M1 |
| U2 | M1 |
| U3 | M2 |
| U4 | M3 |

**Output:**

| MortgageType | RateOfMortgage |
|--------------|---------------:|
| Adjustable | 2.8 |
| Fixed | 4.5 |
| Variable | 3.2 |

**Explanation:** Each mortgage type is matched with its corresponding interest rate using `MortgageID`. Multiple users associated with the same mortgage do not affect the reported rate, so each mortgage type appears only once.

### Constraints

- Handle `NULL` values appropriately.
- Return results matching the expected output schema and order.

In [0]:
mirmortgagedetails_data=[("M1","Fixed",4.5),("M2","Variable",3.2),("M3","Adjustable",2.8)]
mirmortgagedetails_df=spark.createDataFrame(mirmortgagedetails_data,["MortgageID","MortgageType","InterestRate"])
display(mirmortgagedetails_df)

mirusermortgages_data=[("U1","M1"),("U2","M1"),("U3","M2"),("U4","M3")]
mirusermortgages_df=spark.createDataFrame(mirusermortgages_data,["UserID","MortgageID"])
display(mirusermortgages_df)

joined_df = (
mirmortgagedetails_df.alias("d").
    join(mirusermortgages_df.alias("u"), on=col("d.MortgageID") == col("u.MortgageID"), how="left")
)

grouped_df = (
joined_df.groupBy("MortgageType").agg(
    (sum("InterestRate") / count("UserID")).alias("RateOfMortgage")
)
.orderBy("MortgageType"))

display(grouped_df)

MortgageID,MortgageType,InterestRate
M1,Fixed,4.5
M2,Variable,3.2
M3,Adjustable,2.8


UserID,MortgageID
U1,M1
U2,M1
U3,M2
U4,M3


MortgageType,RateOfMortgage
Adjustable,2.8
Fixed,4.5
Variable,3.2


## Busiest Flight Route Analysis

**Difficulty:** Medium

### Problem

Profile name lengths across the flight network for a display-width audit.

You are a data engineer at Air India. The in-flight display team is auditing how much screen space airport names and aircraft model names occupy on boarding screens, so they need the character length of each name associated with every flight. Some name fields carry stray leading or trailing spaces that must not be counted.

Write a query that returns one row per flight in `ba_flights`. Look up the origin airport's name by matching `ba_flights.origin_airport` to `ba_airports.airport_id`, and the destination airport's name by matching `ba_flights.destination_airport` to `ba_airports.airport_id`. Find the aircraft operating each flight by matching `ba_flights.plane_id` to `ba_planes.plane_id`. For each name, trim leading and trailing whitespace first, then return its character length.

**Schema columns:** `ba_flights.flight_id`, `ba_flights.origin_airport`, `ba_flights.destination_airport`, `ba_flights.plane_id`, `ba_airports.airport_id`, `ba_airports.airport_name`, `ba_planes.plane_id`, `ba_planes.plane_model`

**Output columns:** `flight_id`, `origin_airport_name_length`, `destination_airport_name_length`, `plane_model_length`

Order the result by `flight_id` ascending.

### Examples

#### Example 1

**Input:**

**ba_flights:**

| flight_id | origin_airport | destination_airport | plane_id |
|----------:|----------------|---------------------|---------:|
| 1 | A1 | B1 | 2 |
| 2 | A2 | B2 | 3 |
| 3 | A3 | B3 | 1 |

**ba_airports:**

| airport_id | airport_name |
|------------|---------------|
| A1 | San Francisco |
| B1 | Los Angeles |
| A2 | New York |
| B2 | Boston |
| A3 | Miami |
| B3 | Orlando |

**ba_planes:**

| plane_id | plane_model |
|---------:|--------------|
| 1 | Airbus A320 |
| 2 | Boeing 737 |
| 3 | Airbus A380 |

**Output:**

| flight_id | origin_airport_name_length | destination_airport_name_length | plane_model_length |
|----------:|---------------------------:|--------------------------------:|-------------------:|
| 1 | 13 | 11 | 10 |
| 2 | 8 | 6 | 11 |
| 3 | 5 | 7 | 11 |

**Explanation:** Airport names and plane model names are trimmed before calculating their character lengths. Only flights with matching origin airport, destination airport, and plane records are included in the output.

### Constraints

- Join `ba_airports` twice: once for the origin airport and once for the destination airport.
- Match the aircraft using `ba_flights.plane_id = ba_planes.plane_id`.
- Trim leading and trailing whitespace before measuring string length.
- Use inner joins so only fully matched flights are returned.
- Sort the results by `flight_id` in ascending order.

In [0]:
ba_flights_data=[(1,"A1","B1",2),(2,"A2","B2",3),(3,"A3","B3",1)]
ba_flights_df=spark.createDataFrame(ba_flights_data,["flight_id","origin_airport","destination_airport","plane_id"])
display(ba_flights_df)

ba_airports_data=[("A1","         San Francisco   "),("B1","Los Angeles"),("A2","New York   "),("B2","Boston"),("A3","   Miami    "),("B3","Orlando")]
ba_airports_df=spark.createDataFrame(ba_airports_data,["airport_id","airport_name"])
display(ba_airports_df)

ba_planes_data=[(1,"Airbus A320    "),(2,"Boeing 737   "),(3,"   Airbus A380")]
ba_planes_df=spark.createDataFrame(ba_planes_data,["plane_id","plane_model"])
display(ba_planes_df)

joined_df = (
ba_flights_df.alias("f")
    .join(ba_airports_df.alias("oa"), on=col("f.origin_airport") == col("oa.airport_id"), how="inner")
    .join(ba_airports_df.alias("da"), on=col("f.destination_airport") == col("da.airport_id"), how="inner")
    .join(ba_planes_df.alias("p"), on=col("f.plane_id") == col("p.plane_id"), how="inner")
)

output_df = (
joined_df
    .withColumn("origin_airport_name_length", length(trim(col("oa.airport_name"))))
    .withColumn("destination_airport_name_length", length(trim(col("da.airport_name"))))
    .withColumn("plane_model_length", length(trim(col("p.plane_model"))))
    .select("flight_id", "origin_airport_name_length", "destination_airport_name_length", "plane_model_length")
)

display(output_df)

flight_id,origin_airport,destination_airport,plane_id
1,A1,B1,2
2,A2,B2,3
3,A3,B3,1


airport_id,airport_name
A1,San Francisco
B1,Los Angeles
A2,New York
B2,Boston
A3,Miami
B3,Orlando


plane_id,plane_model
1,Airbus A320
2,Boeing 737
3,Airbus A380


flight_id,origin_airport_name_length,destination_airport_name_length,plane_model_length
1,13,11,10
2,8,6,11
3,5,7,11


%md
## CRM Order Summary Report

**Difficulty:** Medium

### Problem

You’re working as a Data Engineer at a company that builds Customer Relationship Management (CRM) software. Your goal is to build a unified view that shows order details along with customer and product information for internal dashboards and reporting.

You are given three datasets that store customer, order, and product details. Write a query that returns each order along with the customer's full name, email, product name, product category, and order date.

**Schema columns:** `crm_customers.customer_id`, `crm_customers.first_name`, `crm_customers.last_name`, `crm_customers.email`, `crm_orders.order_id`, `crm_orders.customer_id`, `crm_orders.product_id`, `crm_orders.order_date`, `crm_products.product_id`, `crm_products.product_name`, `crm_products.category`

**Output columns:** `order_id`, `customer_name`, `customer_email`, `product_name`, `product_category`, `order_date`

Order the result by `order_id` ascending.

### Examples

#### Example 1

**Input:**

**crm_customers:**

| customer_id | first_name | last_name | email |
|------------:|------------|-----------|--------------------|
| 1 | John | Doe | john@example.com |
| 2 | Jane | Smith | jane@example.com |

**crm_orders:**

| order_id | customer_id | product_id | order_date |
|---------:|------------:|-----------:|------------|
| 1001 | 1 | 101 | 2023-01-10 |
| 1002 | 2 | 102 | 2023-01-11 |

**crm_products:**

| product_id | product_name | category |
|-----------:|--------------|-----------|
| 101 | Product A | Category1 |
| 102 | Product B | Category2 |

**Output:**

| order_id | customer_name | customer_email | product_name | product_category | order_date |
|---------:|---------------|----------------|--------------|------------------|------------|
| 1001 | John Doe | john@example.com | Product A | Category1 | 2023-01-10 |
| 1002 | Jane Smith | jane@example.com | Product B | Category2 | 2023-01-11 |

**Explanation:** Each order is joined with its corresponding customer and product. The customer's full name is formed by concatenating the first and last names, and the results are sorted by `order_id`.

### Constraints

- Handle `NULL` values appropriately.
- Return results matching the expected output schema.
- Sort the results by `order_id` in ascending order.

In [0]:
crm_customers_data=[(1,"John","Doe","john@example.com"),(2,"Jane","Smith","jane@example.com")]
crm_customers_df=spark.createDataFrame(crm_customers_data,["customer_id","first_name","last_name","email"])
display(crm_customers_df)

crm_orders_data=[(1001,1,101,"2023-01-10"),(1002,2,102,"2023-01-11")]
crm_orders_df=spark.createDataFrame(crm_orders_data,["order_id","customer_id","product_id","order_date"])

display(crm_orders_df)

crm_products_data=[(101,"Product A","Category1"),(102,"Product B","Category2")]
crm_products_df=spark.createDataFrame(crm_products_data,["product_id","product_name","category"])
display(crm_products_df)

# order_id	customer_name	customer_email	product_name	product_category	order_date
crm_customers_df = (
crm_customers_df
    .withColumn("customer_name", concat_ws(" ", col("first_name"), col("last_name")))
    .withColumnRenamed("email", "customer_email")
    .drop(col("first_name"), col("last_name"))
)


joined_df = (
crm_orders_df.alias("o")
    .join(crm_customers_df.alias("c"), on="customer_id", how="inner")
    .join(crm_products_df.alias("p"), on="product_id", how="inner")
)

output_df = (
joined_df.
    select(col("o.order_id"), col("c.customer_name"), col("c.customer_email"), col("p.product_name"), col("p.category").alias("product_category"), col("o.order_date"))
)

display(output_df)


customer_id,first_name,last_name,email
1,John,Doe,john@example.com
2,Jane,Smith,jane@example.com


order_id,customer_id,product_id,order_date
1001,1,101,2023-01-10
1002,2,102,2023-01-11


product_id,product_name,category
101,Product A,Category1
102,Product B,Category2


order_id,customer_name,customer_email,product_name,product_category,order_date
1001,John Doe,john@example.com,Product A,Category1,2023-01-10
1002,Jane Smith,jane@example.com,Product B,Category2,2023-01-11


%md
## Employee Project Budget Allocation

**Difficulty:** Medium

### Problem

A company tracks each project's budget and the employees assigned to it. For every project that has at least one employee assigned, report the project title, its budget, and the budget available per employee.

In this dataset, each project is worked on by a single employee, so the budget available per employee equals the project's full budget. Projects with no assigned employees should not appear in the result.

**Schema columns:** `apb_employee_table.emp_id`, `apb_employee_table.project_id`, `apb_project_table.id`, `apb_project_table.title`, `apb_project_table.budget`

**Output columns:** `title`, `budget`, `budget_per_employee`

Order the result by `budget` in descending order.

### Examples

#### Example 1

**Input:**

**apb_employee_table:**

| emp_id | project_id |
|-------:|-----------:|
| 10599 | 8 |
| 10606 | 15 |
| 10595 | 4 |

**apb_project_table:**

| id | title | budget |
|---:|---------|-------:|
| 8 | Project8 | 49284 |
| 15 | Project15 | 48116 |
| 4 | Project4 | 15776 |
| 11 | Project11 | 11705 |

**Output:**

| title | budget | budget_per_employee |
|---------|-------:|--------------------:|
| Project8 | 49284 | 49284 |
| Project15 | 48116 | 48116 |
| Project4 | 15776 | 15776 |

**Explanation:** Only projects with at least one assigned employee are included. Since each project has exactly one employee, the budget per employee is equal to the project's total budget. The results are sorted by budget in descending order.

### Constraints

- Include only projects with at least one assigned employee.
- `budget_per_employee` equals the project budget divided by the number of assigned employees.
- In this dataset, each project has one employee, so `budget_per_employee` equals `budget`.
- Sort the results by `budget` in descending order.

In [0]:
apb_employee_table_data=[(10599,8),(10606,15),(10595,4)]
apb_employee_table_df=spark.createDataFrame(apb_employee_table_data,["emp_id","project_id"])
display(apb_employee_table_df)

apb_project_table_data=[(8,"Project8",49284),(15,"Project15",48116),(4,"Project4",15776),(11,"Project11",11705)]
apb_project_table_df=spark.createDataFrame(apb_project_table_data,["id","title","budget"])
display(apb_project_table_df)

joined_df = (
apb_employee_table_df.alias("e").
    join(apb_project_table_df.alias("p"), on=col("e.project_id") == col("p.id"), how="inner")
)

output_df = (
joined_df.select("title", col("budget"), col("budget").alias("budget_per_employee"))
.sort(col("budget").desc())
)

display(output_df)



emp_id,project_id
10599,8
10606,15
10595,4


id,title,budget
8,Project8,49284
15,Project15,48116
4,Project4,15776
11,Project11,11705


title,budget,budget_per_employee
Project8,49284,49284
Project15,48116,48116
Project4,15776,15776


%md
## Product Cross-Sell Pairs

**Difficulty:** Medium

### Problem

An e-commerce platform wants to identify products that are frequently purchased together for recommendations and bundle offers.

Two products form a pair when they appear in the same order. For every such pair, report both product names and `pair_count`, the number of distinct orders in which those two products were purchased together. To avoid duplicate pairs, always use the product with the smaller `product_id` as `product_a` and the larger one as `product_b`. Orders containing only a single product should not contribute any pairs.

**Schema columns:** `order_items.order_id`, `order_items.product_id`, `products.product_id`, `products.product_name`

**Output columns:** `product_a`, `product_b`, `pair_count`

Order the result by `pair_count` descending, then `product_a` ascending, and finally `product_b` ascending.

### Examples

#### Example 1

**Input:**

**order_items:**

| order_id | product_id |
|---------:|-----------:|
| 1 | 1 |
| 1 | 2 |
| 1 | 3 |
| 2 | 1 |
| 2 | 2 |
| 3 | 1 |
| 3 | 3 |
| 4 | 4 |

**products:**

| product_id | product_name |
|-----------:|--------------|
| 1 | Laptop |
| 2 | Mouse |
| 3 | Keyboard |
| 4 | Monitor |

**Output:**

| product_a | product_b | pair_count |
|-----------|-----------|-----------:|
| Laptop | Keyboard | 2 |
| Laptop | Mouse | 2 |
| Mouse | Keyboard | 1 |

**Explanation:** Products that appear together in the same order form a pair. Each pair is counted once per order, using the smaller product ID as `product_a`. Orders containing only one product do not generate any pairs.

### Constraints

- Pair products within the same order so `product_a` has the smaller `product_id`.
- Count the number of distinct orders containing each product pair.
- Report product names instead of product IDs.
- Break ties by `product_a`, then `product_b`, in ascending order.

In [0]:
order_items_data=[(1,1),(1,2),(1,3),(2,1),(2,2),(3,1),(3,3),(4,4)]
order_items_df=spark.createDataFrame(order_items_data,["order_id","product_id"])
display(order_items_df)

products_data=[(1,"Laptop"),(2,"Mouse"),(3,"Keyboard"),(4,"Monitor")]
products_df=spark.createDataFrame(products_data,["product_id","product_name"])
display(products_df)

paired_df = (
order_items_df.alias("o1")
    .join(order_items_df.alias("o2"), on="order_id")
    .where("o1.product_id < o2.product_id")
    .select(col("o1.order_id").alias("order_id"), col("o1.product_id").alias("product_id_a"), col("o2.product_id").alias("product_id_b"))
)

grouped_df = (
paired_df.groupBy("product_id_a", "product_id_b").agg(
    count("order_id").alias("pair_count")
))


joined_df = (
grouped_df.alias("g")
    .join(products_df.alias("p1"), on=col("g.product_id_a") == col("p1.product_id"))
    .join(products_df.alias("p2"), on=col("g.product_id_b") == col("p2.product_id"))
)


output_df = (joined_df
    .select(col("p1.product_name").alias("product_a"), col("p2.product_name").alias("product_b"), col("pair_count"))
    .orderBy(col("pair_count").desc(), col("product_a"), col("product_b"))
)


display(output_df)



order_id,product_id
1,1
1,2
1,3
2,1
2,2
3,1
3,3
4,4


product_id,product_name
1,Laptop
2,Mouse
3,Keyboard
4,Monitor


product_a,product_b,pair_count
Laptop,Keyboard,2
Laptop,Mouse,2
Mouse,Keyboard,1


%md
## Product Groups with No Sales in US

**Difficulty:** Medium

### Problem

Amazon wants to identify product groups that are not performing in the US market. A product group is considered to have **no sales in US** when none of its products has ever been sold in a transaction where the region is `US`.

Write a query to return the name of every such product group.

**Schema columns:** `products.product_id`, `products.group_name`, `products.category`, `sales.sale_id`, `sales.product_id`, `sales.region`, `sales.amount`, `sales.sale_date`

**Output columns:** `group_name`

Order the result by `group_name` ascending.

### Examples

#### Example 1

**Input:**

**products:**

| product_id | group_name | category |
|-----------:|------------|-------------|
| 9 | Books | Fiction |
| 10 | Books | Non-Fiction |
| 13 | Beauty | Skincare |
| 14 | Beauty | Makeup |
| 17 | Furniture | Living Room |
| 18 | Furniture | Bedroom |

**sales:**

| sale_id | product_id | region | amount | sale_date |
|--------:|-----------:|--------|-------:|------------|
| 109 | 9 | EU | 25 | 2024-01-23 |
| 110 | 10 | EU | 35 | 2024-01-24 |
| 113 | 13 | US | 75 | 2024-01-27 |
| 114 | 14 | EU | 95 | 2024-01-28 |
| 117 | 17 | EU | 800 | 2024-01-31 |
| 118 | 18 | ASIA | 950 | 2024-02-01 |

**Output:**

| group_name |
|------------|
| Books |
| Furniture |

**Explanation:** A product group qualifies only if none of its products has a sale with region `US`. Beauty is excluded because one of its products has a US sale, while Books and Furniture have only non-US sales.

### Constraints

- A group qualifies only when none of its products has any sale with region `US`.
- Region matching is exact and case-sensitive on the literal `US`.
- Return each qualifying group name only once.
- Sort the results alphabetically by `group_name`.

In [0]:
products_data=[(9,"Books","Fiction"),(10,"Books","Non-Fiction"),(13,"Beauty","Skincare"),(14,"Beauty","Makeup"),(17,"Furniture","Living Room"),(18,"Furniture","Bedroom")]
products_df=spark.createDataFrame(products_data,["product_id","group_name","category"])
display(products_df)

sales_data=[(109,9,"EU",25,"2024-01-23"),(110,10,"EU",35,"2024-01-24"),(113,13,"US",75,"2024-01-27"),(114,14,"EU",95,"2024-01-28"),(117,17,"EU",800,"2024-01-31"),(118,18,"ASIA",950,"2024-02-01")]
sales_df=spark.createDataFrame(sales_data,["sale_id","product_id","region","amount","sale_date"])
display(sales_df)


us_sales_df = sales_df.filter(col("region") == "US")

us_group = (
products_df.alias("p")
    .join(us_sales_df.alias("s"), on="product_id", how="inner")    
)

non_us_group = (
products_df.alias("p")
    .join(us_group.alias("u"), on="group_name", how="left_anti") 
    .select("group_name")
    .distinct()
    .orderBy("group_name")  
)

display(non_us_group)


product_id,group_name,category
9,Books,Fiction
10,Books,Non-Fiction
13,Beauty,Skincare
14,Beauty,Makeup
17,Furniture,Living Room
18,Furniture,Bedroom


sale_id,product_id,region,amount,sale_date
109,9,EU,25,2024-01-23
110,10,EU,35,2024-01-24
113,13,US,75,2024-01-27
114,14,EU,95,2024-01-28
117,17,EU,800,2024-01-31
118,18,ASIA,950,2024-02-01


group_name
Books
Furniture


%md
## Backfill Missing Dates in Time Series

**Difficulty:** Medium

### Problem

Your analytics team tracks a daily operations metric, but the ingestion process occasionally fails, leaving gaps where some calendar dates have no records. To ensure dashboards display a continuous time series, you need to generate a complete sequence of dates.

Generate a calendar containing every date from the earliest `metric_date` to the latest `metric_date` in `daily_metrics` (inclusive). For each date, return the recorded `metric_value` if it exists. Otherwise, forward-fill the value using the most recent earlier recorded value. Add a `source` column with the value `'actual'` for existing records and `'backfilled'` for generated records.

**Schema columns:** `daily_metrics.metric_date`, `daily_metrics.metric_value`

**Output columns:** `metric_date`, `metric_value`, `source`

Order the result by `metric_date` ascending.

### Examples

#### Example 1

**Input:**

**daily_metrics:**

| metric_date | metric_value |
|-------------|-------------:|
| 2024-01-01 | 100 |
| 2024-01-02 | 105 |
| 2024-01-03 | 110 |
| 2024-01-05 | 120 |
| 2024-01-07 | 130 |
| 2024-01-08 | 135 |

**Output:**

| metric_date | metric_value | source |
|-------------|-------------:|------------|
| 2024-01-01 | 100 | actual |
| 2024-01-02 | 105 | actual |
| 2024-01-03 | 110 | actual |
| 2024-01-04 | 110 | backfilled |
| 2024-01-05 | 120 | actual |
| 2024-01-06 | 120 | backfilled |
| 2024-01-07 | 130 | actual |
| 2024-01-08 | 135 | actual |

**Explanation:** A complete calendar is generated from the minimum to the maximum date. Missing dates are filled with the most recent available metric value and marked as `backfilled`, while existing dates retain their original values and are marked as `actual`.

### Constraints

- Generate every date from the minimum to the maximum `metric_date`, inclusive.
- Missing dates must use the most recent earlier recorded value (forward fill).
- `source` must be exactly `'actual'` or `'backfilled'`.
- The earliest date is guaranteed to exist, so there are no leading gaps.
- Sort the result by `metric_date` in ascending order.

In [0]:
daily_metrics_data=[("2024-01-01",100),("2024-01-02",105),("2024-01-03",110),("2024-01-05",120),("2024-01-07",130),("2024-01-08",135)]
daily_metrics_df=spark.createDataFrame(daily_metrics_data,["metric_date","metric_value"])
display(daily_metrics_df)

start_date = daily_metrics_df.select(min(  col("metric_date").cast("date")  )).collect()[0][0]
end_date = daily_metrics_df.select(max(col("metric_date").cast("date"))).collect()[0][0]


date_df = (
spark.range(1)
.select(explode(sequence(
    lit(start_date),
    lit(end_date)
)).alias("metric_date")
))

joined_df = (
date_df.join(daily_metrics_df, on="metric_date", how="left")
)

window_spec = Window.orderBy("metric_date")

output_df = (
joined_df
    .withColumn("source", 
        when(col("metric_value").isNull(), lit("backfilled")).otherwise(lit("actual"))
    )
    .withColumn("metric_value", 
        when(col("metric_value").isNull(), lag("metric_value").over(window_spec)).otherwise(col("metric_value"))
    )
)

display(output_df)



metric_date,metric_value
2024-01-01,100
2024-01-02,105
2024-01-03,110
2024-01-05,120
2024-01-07,130
2024-01-08,135


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


metric_date,metric_value,source
2024-01-01,100,actual
2024-01-02,105,actual
2024-01-03,110,actual
2024-01-04,110,backfilled
2024-01-05,120,actual
2024-01-06,120,backfilled
2024-01-07,130,actual
2024-01-08,135,actual


%md
## Roblox Daily Active Players by Game Genre

**Difficulty:** Medium

### Problem

You are a data analyst on Roblox's game insights team. Leadership wants a daily view of player engagement by game genre to help guide content strategy and user acquisition.

For each `session_date` and game `genre`, report the number of distinct players who were active in that genre on that day as `unique_players`, counting each player only once even if they had multiple sessions. Also report `total_sessions`, which counts every session for that genre on that day.

Only sessions whose `game_id` matches a game should be included. Name the genre column `game_genre`.

**Schema columns:** `games.game_id`, `games.game_name`, `games.genre`, `sessions.session_id`, `sessions.player_id`, `sessions.game_id`, `sessions.session_date`

**Output columns:** `session_date`, `game_genre`, `unique_players`, `total_sessions`

Order the result by `session_date` ascending, then `game_genre` ascending.

### Examples

#### Example 1

**Input:**

**games:**

| game_id | game_name | genre |
|--------:|-----------|-----------|
| 1 | Adventure Quest | Adventure |
| 2 | Fantasy Realm | Fantasy |

**sessions:**

| session_id | player_id | game_id | session_date |
|-----------:|----------:|--------:|--------------|
| 1 | 101 | 1 | 2024-01-01 |
| 2 | 102 | 1 | 2024-01-01 |
| 3 | 101 | 1 | 2024-01-01 |
| 4 | 103 | 2 | 2024-01-01 |
| 5 | 101 | 2 | 2024-01-02 |
| 6 | 104 | 2 | 2024-01-02 |
| 7 | 105 | 9 | 2024-01-02 |

**Output:**

| session_date | game_genre | unique_players | total_sessions |
|--------------|------------|---------------:|---------------:|
| 2024-01-01 | Adventure | 2 | 3 |
| 2024-01-01 | Fantasy | 1 | 1 |
| 2024-01-02 | Fantasy | 2 | 2 |

**Explanation:** Each player is counted only once per date and genre, while every session contributes to `total_sessions`. Sessions whose `game_id` has no matching record in the `games` table are ignored.

### Constraints

- Include only sessions with a matching `game_id` in the `games` table.
- Count each distinct player once per `session_date` and `game_genre`.
- Count every valid session in `total_sessions`.
- Name the genre column `game_genre`.
- Sort by `session_date`, then `game_genre`, in ascending order.

In [0]:
games_data=[(1,"Adventure Quest","Adventure"),(2,"Fantasy Realm","Fantasy")]
games_df=spark.createDataFrame(games_data,["game_id","game_name","genre"])
display(games_df)

sessions_data=[(1,101,1,"2024-01-01"),(2,102,1,"2024-01-01"),(3,101,1,"2024-01-01"),(4,103,2,"2024-01-01"),(5,101,2,"2024-01-02"),(6,104,2,"2024-01-02"),(7,105,9,"2024-01-02")]
sessions_df=spark.createDataFrame(sessions_data,["session_id","player_id","game_id","session_date"])
display(sessions_df)


grouped_df = (
sessions_df.groupBy("game_id", "session_date").agg(
    countDistinct("player_id").alias("unique_players"),
    count("session_id").alias("total_sessions")
))

joined_df = (
grouped_df.alias("g")
    .join(games_df.alias("p"), on=col("g.game_id") == col("p.game_id"), how="inner")
)

# select session_date	game_genre	unique_players	total_sessions
output_df = joined_df.select("session_date", "genre", "unique_players", "total_sessions").orderBy("session_date", "genre")

display(output_df)


game_id,game_name,genre
1,Adventure Quest,Adventure
2,Fantasy Realm,Fantasy


session_id,player_id,game_id,session_date
1,101,1,2024-01-01
2,102,1,2024-01-01
3,101,1,2024-01-01
4,103,2,2024-01-01
5,101,2,2024-01-02
6,104,2,2024-01-02
7,105,9,2024-01-02


session_date,genre,unique_players,total_sessions
2024-01-01,Adventure,2,3
2024-01-01,Fantasy,1,1
2024-01-02,Fantasy,2,2


%md
## NULL Value Handling in Complex Joins

**Difficulty:** Medium

### Problem

A finance team needs an account summary for every customer, including customers who have never placed an order and orders that have not been paid. Some orders may have multiple partial payments.

For each customer, report the number of orders, the total value of those orders, the total amount paid toward those orders, and the remaining outstanding balance.

`total_orders` counts each order only once. `total_order_amount` sums each order amount once. `total_paid` sums all payments made for the customer's orders. `outstanding_balance` is calculated as `total_order_amount - total_paid`. Use `0` for missing totals and round all monetary values to 2 decimal places. Orders without a customer should not be assigned to any customer.

**Schema columns:** `customers.customer_id`, `customers.customer_name`, `customers.email`, `orders.order_id`, `orders.customer_id`, `orders.order_date`, `orders.amount`, `payments.payment_id`, `payments.order_id`, `payments.payment_date`, `payments.payment_amount`

**Output columns:** `customer_id`, `customer_name`, `total_orders`, `total_order_amount`, `total_paid`, `outstanding_balance`

Order the result by `customer_id` ascending.

### Examples

#### Example 1

**Input:**

**customers:**

| customer_id | customer_name | email |
|------------:|---------------|----------------------|
| 1001 | Customer_1001 | [email protected] |
| 1002 | Customer_1002 | [email protected] |
| 1006 | Customer_1006 | [email protected] |

**orders:**

| order_id | customer_id | order_date | amount |
|---------:|------------:|------------|-------:|
| 20022 | 1001 | 2024-09-09 | 984.64 |
| 20025 | 1001 | 2024-10-15 | 1662.94 |
| 20028 | 1006 | 2024-11-20 | 1283.49 |

**payments:**

| payment_id | order_id | payment_date | payment_amount |
|-----------:|---------:|--------------|---------------:|
| 30003 | 20028 | 2024-01-31 | 1086.16 |
| 30013 | 20025 | 2024-06-29 | 1007.29 |

**Output:**

| customer_id | customer_name | total_orders | total_order_amount | total_paid | outstanding_balance |
|------------:|---------------|-------------:|-------------------:|-----------:|--------------------:|
| 1001 | Customer_1001 | 2 | 2647.58 | 1007.29 | 1640.29 |
| 1002 | Customer_1002 | 0 | 0.00 | 0.00 | 0.00 |
| 1006 | Customer_1006 | 1 | 1283.49 | 1086.16 | 197.33 |

**Explanation:** Every customer appears in the result, even if they have no orders. Order totals count each order only once, while payment totals include every payment made toward the customer's orders. Customers with no orders receive zero values for all totals.

### Constraints

- Return every customer exactly once, including customers with no orders.
- Count and sum each order only once, even if it has multiple payments.
- Include all payments associated with a customer's orders.
- Ignore orders with no associated customer.
- Round all monetary values to 2 decimal places.
- Sort the results by `customer_id` in ascending order.

In [0]:
customers_data=[(1001,"Customer_1001","[email protected]"),(1002,"Customer_1002","[email protected]"),(1006,"Customer_1006","[email protected]")]
customers_df=spark.createDataFrame(customers_data,["customer_id","customer_name","email"])
display(customers_df)

orders_data=[(20022,1001,"2024-09-09",984.64),(20025,1001,"2024-10-15",1662.94),(20028,1006,"2024-11-20",1283.49)]
orders_df=spark.createDataFrame(orders_data,["order_id","customer_id","order_date","amount"])
display(orders_df)

payments_data=[(30003,20028,"2024-01-31",1086.16),(30013,20025,"2024-06-29",1007.29)]
payments_df=spark.createDataFrame(payments_data,["payment_id","order_id","payment_date","payment_amount"])
display(payments_df)

total_payment = (
payments_df.groupBy("order_id").agg(sum("payment_amount").alias("total_paid"))
)

order_total_payment_df = (
orders_df.alias("o")
    .join(total_payment.alias("t"), on="order_id", how="left")
    .fillna(0, subset=["total_paid"])
)

# total_orders	total_order_amount	total_paid	outstanding_balance

order_total_payment_grouped_df = (
order_total_payment_df.groupBy("customer_id").agg(
    count("o.order_id").alias("total_order"),
    sum("amount").alias("total_order_amount"),
    sum("total_paid").alias("total_paid"),
     round(col("total_order_amount") - col("total_paid"), 2).alias("outstanding_balance2")
    )
    .withColumn("outstanding_balance", round(col("total_order_amount") - col("total_paid"), 2))
)


output_df = (
customers_df.alias("c")
    .join(order_total_payment_grouped_df.alias("o"), on="customer_id", how="left")
    .select("c.customer_id","c.customer_name","o.total_order","o.total_order_amount","o.total_paid","o.outstanding_balance")
    .fillna(0)
    .orderBy("c.customer_id")
)


display(output_df)




customer_id,customer_name,email
1001,Customer_1001,[email protected]
1002,Customer_1002,[email protected]
1006,Customer_1006,[email protected]


order_id,customer_id,order_date,amount
20022,1001,2024-09-09,984.64
20025,1001,2024-10-15,1662.94
20028,1006,2024-11-20,1283.49


payment_id,order_id,payment_date,payment_amount
30003,20028,2024-01-31,1086.16
30013,20025,2024-06-29,1007.29


customer_id,customer_name,total_order,total_order_amount,total_paid,outstanding_balance
1001,Customer_1001,2,2647.58,1007.29,1640.29
1002,Customer_1002,0,0.0,0.0,0.0
1006,Customer_1006,1,1283.49,1086.16,197.33


%md
## Cross Join and Cartesian Products

**Difficulty:** Medium

### Problem

An apparel store wants to generate a complete product catalog containing every possible combination of product, size, and color. Each combination represents a unique product variant that can be purchased.

Create one row for every combination of product, size, and color. For each combination, return the product name, size name, color name, and a SKU in the format `product_id-size_id-color_id`.

**Schema columns:** `products.product_id`, `products.product_name`, `sizes.size_id`, `sizes.size_name`, `colors.color_id`, `colors.color_name`

**Output columns:** `product_name`, `size_name`, `color_name`, `sku`

Order the result by `product_name` ascending, `size_name` ascending, and `color_name` ascending.

### Examples

#### Example 1

**Input:**

**products:**

| product_id | product_name |
|-----------:|--------------|
| 1 | T-Shirt |
| 2 | Jeans |

**sizes:**

| size_id | size_name |
|--------:|-----------|
| 1 | S |
| 2 | M |

**colors:**

| color_id | color_name |
|---------:|------------|
| 1 | Red |
| 2 | Blue |

**Output:**

| product_name | size_name | color_name | sku |
|--------------|-----------|------------|---------|
| Jeans | M | Blue | 2-2-2 |
| Jeans | M | Red | 2-2-1 |
| Jeans | S | Blue | 2-1-2 |
| Jeans | S | Red | 2-1-1 |
| T-Shirt | M | Blue | 1-2-2 |
| T-Shirt | M | Red | 1-2-1 |
| T-Shirt | S | Blue | 1-1-2 |
| T-Shirt | S | Red | 1-1-1 |

**Explanation:** Every product is paired with every size and every color, producing all possible combinations. The SKU is created by concatenating `product_id`, `size_id`, and `color_id` with hyphens.

### Constraints

- Generate every possible product-size-color combination exactly once.
- Create the SKU in the format `product_id-size_id-color_id`.
- Return the product name, size name, color name, and SKU.
- Sort the result by `product_name`, `size_name`, and `color_name` in ascending order.

In [0]:
products_data=[(1,"T-Shirt"),(2,"Jeans")]
products_df=spark.createDataFrame(products_data,["product_id","product_name"])
display(products_df)

sizes_data=[(1,"S"),(2,"M")]
sizes_df=spark.createDataFrame(sizes_data,["size_id","size_name"])
display(sizes_df)

colors_data=[(1,"Red"),(2,"Blue")]
colors_df=spark.createDataFrame(colors_data,["color_id","color_name"])
display(colors_df)


cartesian_df = (
products_df
    .crossJoin(sizes_df)
    .crossJoin(colors_df)
    .withColumn("sku", concat_ws("-", col("product_id"), col("size_id"), col("color_id")))
    .select("product_name", "size_name", "color_name", "sku")
    .orderBy("product_name", "size_name", "color_name")
)

display(cartesian_df)



product_id,product_name
1,T-Shirt
2,Jeans


size_id,size_name
1,S
2,M


color_id,color_name
1,Red
2,Blue


product_name,size_name,color_name,sku
Jeans,M,Blue,2-2-2
Jeans,M,Red,2-2-1
Jeans,S,Blue,2-1-2
Jeans,S,Red,2-1-1
T-Shirt,M,Blue,1-2-2
T-Shirt,M,Red,1-2-1
T-Shirt,S,Blue,1-1-2
T-Shirt,S,Red,1-1-1


%md
## Account Balance Segmentation

**Difficulty:** Medium

### Problem

Write a query to categorize bank accounts based on their monthly salary.

- **Low Income** → Monthly salary is strictly less than `20000`.
- **Medium Income** → Monthly salary is between `20000` and `50000` (inclusive).
- **High Income** → Monthly salary is strictly greater than `50000`.

Return the number of accounts in each category. If a category has no accounts, its count should still be returned as `0`.

**Schema columns:** `bas_bank_accounts.account_id`, `bas_bank_accounts.monthly_income`

**Output columns:** `category`, `accounts_count`

Order the result as `High Income`, `Medium Income`, then `Low Income`.

### Examples

#### Example 1

**Input:**

**bas_bank_accounts:**

| account_id | monthly_income |
|-----------:|---------------:|
| 3 | 108939 |
| 2 | 12747 |
| 8 | 87709 |
| 6 | 91796 |

**Output:**

| category | accounts_count |
|-----------|---------------:|
| High Income | 3 |
| Medium Income | 0 |
| Low Income | 1 |

**Explanation:** Three accounts have a monthly income greater than 50000 and belong to the **High Income** category. One account has a monthly income below 20000 and belongs to the **Low Income** category. No accounts fall into the **Medium Income** category, so its count is returned as 0.

### Constraints

- Handle NULL values appropriately.
- Return all three income categories, even if a category has zero accounts.
- Categorize accounts using the specified income ranges.
- Return results matching the expected output schema and order.

In [0]:
bas_bank_accounts_data=[(3,108939),(2,12747),(8,87709),(6,91796)]
bas_bank_accounts_df=spark.createDataFrame(bas_bank_accounts_data,["account_id","monthly_income"])
display(bas_bank_accounts_df)

category_df = spark.createDataFrame([("Low Income", 0),("Medium Income", 0),("High Income", 0)], ["category", "count"])

salary_df = (
bas_bank_accounts_df.withColumn("category",
        when(col("monthly_income") < 20000, "Low Income")
        .when(col("monthly_income") <= 50000, "Medium Income")
        .otherwise("High Income")
    )
)

grouped_df = salary_df.groupBy("category").agg(count("account_id").alias("count")).union(category_df)

output_df = (
grouped_df
.groupBy("category").agg(sum("count").alias("count"))
.withColumn("priority", when(col("category") == "Low Income", 0).when(col("category") == "Medium Income", 1).otherwise(2))
.orderBy(col("priority").desc())
.select("category", "count")
)

display(output_df)


account_id,monthly_income
3,108939
2,12747
8,87709
6,91796


category,count
High Income,3
Medium Income,0
Low Income,1


%md
## Find Mutual Friends Between Two Users

**Difficulty:** Medium

### Problem

A social network stores each user's friends as `(user_id, friend_id)`, where `friend_id` belongs to that user's friend list.

Write a query to find the mutual friends between **user 1** and **user 2**. Return the distinct friend IDs that appear in both users' friend lists.

**Schema columns:** `friendships.user_id`, `friendships.friend_id`

**Output columns:** `mutual_friend_id`

Order the result by `mutual_friend_id` ascending.

### Examples

#### Example 1

**Input:**

**friendships:**

| user_id | friend_id |
|--------:|----------:|
| 1 | 2 |
| 1 | 3 |
| 1 | 4 |
| 1 | 5 |
| 2 | 1 |
| 2 | 3 |
| 2 | 5 |
| 2 | 6 |

**Output:**

| mutual_friend_id |
|-----------------:|
| 3 |
| 5 |

**Explanation:** User 1's friend list is `{2, 3, 4, 5}` and user 2's friend list is `{1, 3, 5, 6}`. The common friends in both lists are `3` and `5`.

### Constraints

- Return each mutual friend only once.
- Consider only the friend lists of user `1` and user `2`.
- Sort the result by `mutual_friend_id` in ascending order.
- Return results matching the expected output schema.

In [0]:
friendships_data=[(1,2),(1,3),(1,4),(1,5),(2,1),(2,3),(2,5),(2,6)]
friendships_df=spark.createDataFrame(friendships_data,["user_id","friend_id"])
display(friendships_df)

user1_df = friendships_df.filter(col("user_id") == 1)
user2_df = friendships_df.filter(col("user_id") == 2)

joined_df = (
    user1_df.join(user2_df, on="friend_id", how="inner")
)

output_df = joined_df.select("friend_id").distinct()

display(output_df)

# output_df = (
# user1_df.intersect(user2_df)
# .distinct()
# )

# display(output_df)



user_id,friend_id
1,2
1,3
1,4
1,5
2,1
2,3
2,5
2,6


friend_id
3
5


%md
## Symmetric Pairs in Functions Table

**Difficulty:** Medium

### Problem

A pair `(x, y)` is called **symmetric** if the reverse pair `(y, x)` also exists in the table.

Write a query to return every symmetric pair exactly once in the orientation where `x <= y`. A self-pair such as `(3, 3)` is considered symmetric by itself.

**Schema columns:** `functions.x`, `functions.y`

**Output columns:** `x`, `y`

Order the result by `x` ascending.

### Examples

#### Example 1

**Input:**

**functions:**

| x | y |
|--:|--:|
| 1 | 2 |
| 2 | 1 |
| 3 | 4 |
| 4 | 3 |
| 5 | 5 |
| 6 | 6 |
| 7 | 8 |
| 8 | 9 |

**Output:**

| x | y |
|--:|--:|
| 1 | 2 |
| 3 | 4 |
| 5 | 5 |
| 6 | 6 |

**Explanation:** The pairs `(1,2)` and `(2,1)` form one symmetric pair, so only `(1,2)` is returned. Similarly, `(3,4)` and `(4,3)` produce `(3,4)`. Self-pairs `(5,5)` and `(6,6)` qualify on their own because the reverse pair is identical.

### Constraints

- Return each symmetric pair only once.
- Output each pair in the orientation `x <= y`.
- Self-pairs qualify as symmetric.
- Sort the result by `x` in ascending order.

In [0]:
functions_data=[(1,2),(1,2),(2,1),(3,4),(4,3),(5,5),(6,6),(7,8),(8,9)]
functions_df=spark.createDataFrame(functions_data,["x","y"])
display(functions_df)

output_df = (
functions_df.alias("x")
    .join(functions_df.alias("y"), on=((col("x.x") == col("y.y")) & (col("x.y") == col("y.x"))))
    .filter("x.x <= x.y")
    .select("x.*")
    .distinct()
    .orderBy("x.x")
)

display(output_df)



x,y
1,2
1,2
2,1
3,4
4,3
5,5
6,6
7,8
8,9


x,y
1,2
3,4
5,5
6,6


%md
## Employees Earning More Than Their Manager

**Difficulty:** Medium

### Problem

A company wants to identify employees whose annual salary is strictly greater than the salary of their direct manager.

Each employee record contains the employee's ID, manager's ID, name, and annual salary. Return the name of every employee who earns more than their manager, along with the manager's name.

Employees who do not have a manager should not be included.

**Schema columns:** `employees.employee_id`, `employees.manager_id`, `employees.name`, `employees.salary`

**Output columns:** `employee_name`, `manager_name`

Order the result by `employee_id` ascending.

### Examples

#### Example 1

**Input:**

**employees:**

| employee_id | manager_id | name | salary |
|------------:|-----------:|------|-------:|
| 101 | NULL | CEO | 100000 |
| 102 | 101 | Alice | 80000 |
| 103 | 101 | Bob | 110000 |
| 104 | 102 | Charlie | 85000 |
| 105 | 102 | Diana | 70000 |

**Output:**

| employee_name | manager_name |
|---------------|--------------|
| Bob | CEO |
| Charlie | Alice |

**Explanation:** Bob earns 110000, which is greater than his manager CEO's salary of 100000. Charlie earns 85000, which is greater than his manager Alice's salary of 80000. Alice and Diana earn less than their managers, while the CEO has no manager and is therefore excluded.

### Constraints

- Do not include employees with a NULL `manager_id`.
- Return employees whose salary is strictly greater than their direct manager's salary.
- Return the employee name and manager name.
- Sort the results by `employee_id` in ascending order.

In [0]:
employees_data=[(101,None,"CEO",100000),(102,101,"Alice",80000),(103,101,"Bob",110000),(104,102,"Charlie",85000),(105,102,"Diana",70000)]
employees_df=spark.createDataFrame(employees_data,["employee_id","manager_id","name","salary"])
display(employees_df)

joined_df = (
employees_df.alias("e")
    .join(employees_df.alias("m"), on=(col("e.manager_id") == col("m.employee_id") )& (col("e.salary") > col("m.salary")))
    .orderBy("e.employee_id")
    .select(col("e.name").alias("employee_name"), col("m.name").alias("manager_name"))
)

display(joined_df)



employee_id,manager_id,name,salary
101,null,CEO,100000
102,101,Alice,80000
103,101,Bob,110000
104,102,Charlie,85000
105,102,Diana,70000


employee_name,manager_name
Bob,CEO
Charlie,Alice


%md
## Self-Join for Pair Comparisons

**Difficulty:** Medium

### Problem

An HR team needs a list of direct employee-manager relationships. Each employee record contains the employee's department, salary, and the ID of their direct manager.

For every employee who has a valid manager, return the employee's name, manager's name, employee's department, and the salary difference, where:

`salary_diff = manager_salary - employee_salary`

Only direct reporting relationships should be included.

**Schema columns:** `employees.employee_id`, `employees.name`, `employees.department`, `employees.salary`, `employees.manager_id`

**Output columns:** `employee_name`, `manager_name`, `department`, `salary_diff`

Order the result by `employee_name` ascending.

### Examples

#### Example 1

**Input:**

**employees:**

| employee_id | name | department | salary | manager_id |
|------------:|------|------------|-------:|-----------:|
| 1 | Alice | Sales | 60000 | NULL |
| 2 | Bob | Sales | 55000 | 1 |
| 3 | Charlie | IT | 75000 | NULL |
| 4 | David | IT | 78000 | 3 |
| 5 | Eve | HR | 65000 | NULL |

**Output:**

| employee_name | manager_name | department | salary_diff |
|---------------|--------------|------------|------------:|
| Bob | Alice | Sales | 5000 |
| David | Charlie | IT | -3000 |

**Explanation:** Bob reports directly to Alice, so the salary difference is `60000 - 55000 = 5000`. David reports directly to Charlie, so the salary difference is `75000 - 78000 = -3000`. Employees without managers are not included.

### Constraints

- Return only direct employee-manager relationships.
- An employee cannot be their own manager.
- Calculate `salary_diff` as `manager_salary - employee_salary`.
- Sort the result by `employee_name` in ascending order.

In [0]:
employees_data=[(1,"Alice","Sales",60000,None),(2,"Bob","Sales",55000,1),(3,"Charlie","IT",75000,None),(4,"David","IT",78000,3),(5,"Eve","HR",65000,None)]

employees_df=spark.createDataFrame(employees_data,["employee_id","name","department","salary","manager_id"])


joined_df = (
employees_df.alias("e")
    .join(employees_df.alias("m"), on=col("e.manager_id") == col("m.employee_id") )
)

# employee_name	manager_name	department	salary_diff
output_df = (
    joined_df
    .select(col("e.name").alias("employee_name"), col("m.name").alias("manager_name"), col("e.department"), (col("m.salary") - col("e.salary")).alias("salary_diff"))
    .orderBy("employee_name")
)

display(output_df)


employee_name,manager_name,department,salary_diff
Bob,Alice,Sales,5000
David,Charlie,IT,-3000


%md
## PySpark Broadcast Join Optimization

**Difficulty:** Hard

### Problem

A payments team wants to summarize transactions by continent using the country associated with each transaction.

For each continent, return:
- The total number of transactions.
- The total transaction amount rounded to 2 decimal places.
- The number of distinct users who made transactions.
- The average transaction amount rounded to 2 decimal places.

**Schema columns:** `countries.country_code`, `countries.country_name`, `countries.continent`, `transactions.txn_id`, `transactions.user_id`, `transactions.country_code`, `transactions.amount`, `transactions.txn_date`

**Output columns:** `continent`, `total_transactions`, `total_amount`, `unique_users`, `avg_amount`

Order the result by `total_amount` in descending order. If multiple continents have the same total amount, they may be returned in any order.

### Examples

#### Example 1

**Input:**

**countries:**

| country_code | country_name | continent |
|--------------|--------------|-----------|
| US | United States | North America |
| FR | France | Europe |
| DE | Germany | Europe |

**transactions:**

| txn_id | user_id | country_code | amount | txn_date |
|-------:|--------:|--------------|-------:|------------|
| 1 | 10 | US | 100 | 2024-01-01 |
| 2 | 11 | FR | 70 | 2024-01-02 |
| 3 | 11 | DE | 30 | 2024-01-03 |

**Output:**

| continent | total_transactions | total_amount | unique_users | avg_amount |
|-----------|-------------------:|-------------:|-------------:|-----------:|
| Europe | 2 | 100.00 | 1 | 50.00 |
| North America | 1 | 100.00 | 1 | 100.00 |

**Explanation:** The transactions from France and Germany both belong to Europe, resulting in 2 transactions with a total amount of 100.00. Since both transactions were made by the same user, Europe has 1 unique user and an average transaction amount of 50.00. North America has one transaction worth 100.00 made by one user.

### Constraints

- Every transaction references a valid country code.
- Count each user only once per continent when calculating `unique_users`.
- Round `total_amount` and `avg_amount` to 2 decimal places.
- Order by `total_amount` in descending order. Continents with the same total amount may appear in any order.

In [0]:
countries_data=[("US","United States","North America"),("FR","France","Europe"),("DE","Germany","Europe")]
countries_df=spark.createDataFrame(countries_data,["country_code","country_name","continent"])
display(countries_df)

transactions_data=[(1,10,"US",100,"2024-01-01"),(2,11,"FR",70,"2024-01-02"),(3,11,"DE",30,"2024-01-03")]
transactions_df=spark.createDataFrame(transactions_data,["txn_id","user_id","country_code","amount","txn_date"])
display(transactions_df)


joined_df = (
transactions_df.alias("t").
    join(broadcast(countries_df).alias("c"), on=col("t.country_code") == col("c.country_code"), how="inner")
)

# continent	total_transactions	total_amount	unique_users	avg_amount
grouped_df = (
    joined_df.groupBy("continent").agg(
        count("txn_id").alias("total_transactions"),
        sum("amount").alias("total_amount"),
        countDistinct("user_id").alias("unique_users"),
        avg("amount").alias("avg_amount")
    )
    .orderBy("total_amount")

)


display(grouped_df)


country_code,country_name,continent
US,United States,North America
FR,France,Europe
DE,Germany,Europe


txn_id,user_id,country_code,amount,txn_date
1,10,US,100,2024-01-01
2,11,FR,70,2024-01-02
3,11,DE,30,2024-01-03


continent,total_transactions,total_amount,unique_users,avg_amount
Europe,2,100,1,50.0
North America,1,100,1,100.0


%md
## Recursive CTE for Organizational Hierarchy

**Difficulty:** Hard

### Problem

You are given a company's employee table where each employee references their direct manager through `manager_id`. The CEO is the only employee without a manager.

For every employee, return the employee ID, employee name, direct manager ID, direct manager name, the employee's level in the organization, and the complete reporting path from the CEO to that employee.

The hierarchy level starts at **1** for the CEO and increases by **1** for each reporting level below.

**Schema columns:** `employees.emp_id`, `employees.emp_name`, `employees.manager_id`, `employees.salary`

**Output columns:** `emp_id`, `emp_name`, `manager_id`, `manager_name`, `hierarchy_level`, `path`

Order the result by `emp_id` ascending.

### Examples

#### Example 1

**Input:**

**employees:**

| emp_id | emp_name | manager_id | salary |
|-------:|-----------|-----------:|-------:|
| 1 | CEO Alice | NULL | 250000 |
| 2 | VP Bob | 1 | 180000 |
| 3 | VP Charlie | 1 | 175000 |
| 4 | Manager David | 2 | 120000 |
| 6 | Manager Frank | 3 | 110000 |
| 7 | Engineer Grace | 4 | 90000 |

**Output:**

| emp_id | emp_name | manager_id | manager_name | hierarchy_level | path |
|-------:|-----------|-----------:|--------------|----------------:|------|
| 1 | CEO Alice | NULL | NULL | 1 | CEO Alice |
| 2 | VP Bob | 1 | CEO Alice | 2 | CEO Alice > VP Bob |
| 3 | VP Charlie | 1 | CEO Alice | 2 | CEO Alice > VP Charlie |
| 4 | Manager David | 2 | VP Bob | 3 | CEO Alice > VP Bob > Manager David |
| 6 | Manager Frank | 3 | VP Charlie | 3 | CEO Alice > VP Charlie > Manager Frank |
| 7 | Engineer Grace | 4 | Manager David | 4 | CEO Alice > VP Bob > Manager David > Engineer Grace |

**Explanation:** The CEO is the root of the hierarchy and has no manager. Each employee's hierarchy level is determined by their distance from the CEO, and the `path` contains the complete chain of employee names from the CEO to that employee.

### Constraints

- The CEO is the only employee with a NULL `manager_id`.
- The CEO has a `hierarchy_level` of 1 and no manager name.
- `hierarchy_level` increases by 1 for each level in the reporting chain.
- `path` contains the employee names from the CEO to the current employee, separated by `>`.
- Sort the results by `emp_id` in ascending order.

In [0]:
employees_data=[(1,"CEO Alice",None,250000),(2,"VP Bob",1,180000),(3,"VP Charlie",1,175000),(4,"Manager David",2,120000),(6,"Manager Frank",3,110000),(7,"Engineer Grace",4,90000)]
employees_df=spark.createDataFrame(employees_data,["emp_id","emp_name","manager_id","salary"])
display(employees_df)



emp_id,emp_name,manager_id,salary
1,CEO Alice,null,250000
2,VP Bob,1,180000
3,VP Charlie,1,175000
4,Manager David,2,120000
6,Manager Frank,3,110000
7,Engineer Grace,4,90000


%md
## SKU-Level Return Rates

**Difficulty:** Hard

### Problem

A marketplace wants to analyze product return rates at the SKU level.

For each SKU that has sold at least **5 units**, return the total units sold, total units returned, and the return rate. If a SKU has no returns, its total returned should be reported as **0**.

The return rate is calculated as:

`total_returned / total_sold`

and rounded to **2 decimal places**.

**Schema columns:** `order_items.order_id`, `order_items.sku_id`, `order_items.quantity`, `order_items.price`, `order_items.order_date`, `returns.return_id`, `returns.order_id`, `returns.sku_id`, `returns.quantity_returned`, `returns.return_date`, `returns.reason`

**Output columns:** `sku_id`, `total_sold`, `total_returned`, `return_rate`

Order the result by `return_rate` in descending order, then by `sku_id` in ascending order.

### Examples

#### Example 1

**Input:**

**order_items:**

| order_id | sku_id | quantity | price | order_date |
|---------:|-------:|---------:|------:|------------|
| 1 | 101 | 2 | 29.99 | 2024-01-05 |
| 2 | 102 | 1 | 49.99 | 2024-01-06 |
| 3 | 103 | 3 | 15.99 | 2024-01-07 |
| 4 | 101 | 1 | 29.99 | 2024-01-08 |
| 5 | 104 | 2 | 89.99 | 2024-01-09 |
| 6 | 102 | 2 | 49.99 | 2024-01-10 |
| 7 | 105 | 1 | 19.99 | 2024-01-11 |
| 8 | 101 | 3 | 29.99 | 2024-01-12 |
| 9 | 103 | 2 | 15.99 | 2024-01-13 |
| 10 | 106 | 5 | 9.99 | 2024-01-14 |
| 11 | 102 | 1 | 49.99 | 2024-01-15 |
| 12 | 104 | 1 | 89.99 | 2024-01-16 |
| 13 | 107 | 2 | 39.99 | 2024-01-17 |
| 14 | 101 | 2 | 29.99 | 2024-01-18 |
| 15 | 103 | 4 | 15.99 | 2024-01-19 |
| 16 | 102 | 3 | 49.99 | 2024-01-20 |
| 17 | 108 | 1 | 59.99 | 2024-01-21 |
| 18 | 104 | 2 | 89.99 | 2024-01-22 |
| 19 | 105 | 2 | 19.99 | 2024-01-23 |
| 20 | 106 | 3 | 9.99 | 2024-01-24 |
| 21 | 103 | 1 | 15.99 | 2024-01-25 |
| 22 | 101 | 1 | 29.99 | 2024-01-26 |
| 23 | 102 | 2 | 49.99 | 2024-01-27 |
| 24 | 107 | 1 | 39.99 | 2024-01-28 |
| 25 | 104 | 3 | 89.99 | 2024-01-29 |
| 26 | 109 | 2 | 74.99 | 2024-01-30 |
| 27 | 106 | 2 | 9.99 | 2024-01-31 |

**returns:**

| return_id | order_id | sku_id | quantity_returned | return_date | reason |
|----------:|---------:|-------:|------------------:|------------|------------------|
| 1 | 1 | 101 | 1 | 2024-01-10 | defective |
| 2 | 3 | 103 | 1 | 2024-01-12 | wrong_size |
| 3 | 6 | 102 | 1 | 2024-01-18 | not_as_described |
| 4 | 8 | 101 | 2 | 2024-01-20 | damaged |
| 5 | 9 | 103 | 1 | 2024-01-22 | defective |
| 6 | 10 | 106 | 2 | 2024-01-25 | wrong_color |
| 7 | 14 | 101 | 1 | 2024-01-28 | damaged |
| 8 | 16 | 102 | 2 | 2024-01-30 | not_as_described |
| 9 | 15 | 103 | 2 | 2024-02-02 | wrong_size |
| 10 | 18 | 104 | 1 | 2024-02-05 | defective |
| 11 | 20 | 106 | 1 | 2024-02-08 | damaged |
| 12 | 25 | 104 | 1 | 2024-02-12 | not_as_described |

**Output:**

| sku_id | total_sold | total_returned | return_rate |
|-------:|-----------:|---------------:|------------:|
| 101 | 9 | 4 | 0.44 |
| 103 | 10 | 4 | 0.40 |
| 102 | 9 | 3 | 0.33 |
| 106 | 10 | 3 | 0.30 |
| 104 | 8 | 2 | 0.25 |

**Explanation:** Only SKUs with at least 5 units sold are included. For example, SKU 101 sold 9 units and had 4 returned units, giving a return rate of `4 / 9 = 0.44` after rounding.

### Constraints

- Include only SKUs with `total_sold >= 5`.
- `total_returned` is the sum of `quantity_returned`, or `0` if no returns exist.
- Round `return_rate` to 2 decimal places.
- Order by `return_rate` descending, then `sku_id` ascending.

In [0]:
order_items_data=[(1,101,2,29.99,"2024-01-05"),(2,102,1,49.99,"2024-01-06"),(3,103,3,15.99,"2024-01-07"),(4,101,1,29.99,"2024-01-08"),(5,104,2,89.99,"2024-01-09"),(6,102,2,49.99,"2024-01-10"),(7,105,1,19.99,"2024-01-11"),(8,101,3,29.99,"2024-01-12"),(9,103,2,15.99,"2024-01-13"),(10,106,5,9.99,"2024-01-14"),(11,102,1,49.99,"2024-01-15"),(12,104,1,89.99,"2024-01-16"),(13,107,2,39.99,"2024-01-17"),(14,101,2,29.99,"2024-01-18"),(15,103,4,15.99,"2024-01-19"),(16,102,3,49.99,"2024-01-20"),(17,108,1,59.99,"2024-01-21"),(18,104,2,89.99,"2024-01-22"),(19,105,2,19.99,"2024-01-23"),(20,106,3,9.99,"2024-01-24"),(21,103,1,15.99,"2024-01-25"),(22,101,1,29.99,"2024-01-26"),(23,102,2,49.99,"2024-01-27"),(24,107,1,39.99,"2024-01-28"),(25,104,3,89.99,"2024-01-29"),(26,109,2,74.99,"2024-01-30"),(27,106,2,9.99,"2024-01-31")]
order_items_df=spark.createDataFrame(order_items_data,["order_id","sku_id","quantity","price","order_date"])

returns_data=[(1,1,101,1,"2024-01-10","defective"),(2,3,103,1,"2024-01-12","wrong_size"),(3,6,102,1,"2024-01-18","not_as_described"),(4,8,101,2,"2024-01-20","damaged"),(5,9,103,1,"2024-01-22","defective"),(6,10,106,2,"2024-01-25","wrong_color"),(7,14,101,1,"2024-01-28","damaged"),(8,16,102,2,"2024-01-30","not_as_described"),(9,15,103,2,"2024-02-02","wrong_size"),(10,18,104,1,"2024-02-05","defective"),(11,20,106,1,"2024-02-08","damaged"),(12,25,104,1,"2024-02-12","not_as_described")]
returns_df=spark.createDataFrame(returns_data,["return_id","order_id","sku_id","quantity_returned","return_date","reason"])


grouped_orders_df = (
order_items_df.groupBy("sku_id").agg(
    sum("quantity").alias("total_sold")
)
.filter(col("total_sold") >= 5)

)

grouped_return_df = (
returns_df.groupBy("sku_id").agg(
    sum("quantity_returned").alias("total_returned")
))


result_df = (
grouped_orders_df.alias("o").join(
    grouped_return_df.alias("r"),on="sku_id",how="left"
)
.select(col("sku_id"),col("total_sold"),
    coalesce(col("total_returned"), lit(0)).alias("total_returned")
)
.withColumn("return_rate", round(col("total_returned") / col("total_sold"), 2))
.orderBy(col("return_rate").desc(),col("sku_id").asc()
))

display(result_df)


sku_id,total_sold,total_returned,return_rate
101,9,4,0.44
103,10,4,0.4
102,9,3,0.33
106,10,3,0.3
104,8,2,0.25


%md
## Advertiser Payment Status Classification

**Difficulty:** Hard

### Problem

An advertising platform updates each advertiser's account status based on yesterday's advertiser status and today's payment activity.

For every user appearing in either the `advertiser` table or the `daily_pay` table, determine the user's new status according to the following rules:

- A user with a prior status of `CHURN` who makes a payment becomes `RESURRECT`.
- A user with a prior status of `NEW`, `EXISTING`, or `RESURRECT` who makes a payment becomes `EXISTING`.
- A user with a prior status of `NEW`, `EXISTING`, or `RESURRECT` who does not make a payment becomes `CHURN`.
- A user who makes a payment but has no previous advertiser record becomes `NEW`.

**Schema columns:** `advertiser.user_id`, `advertiser.status`, `daily_pay.user_id`, `daily_pay.paid`

**Output columns:** `user_id`, `new_status`

Order the result by `user_id` in ascending order.

### Examples

#### Example 1

**Input:**

**advertiser:**

| user_id | status |
|--------:|----------|
| 1 | NEW |
| 2 | EXISTING |
| 3 | CHURN |
| 4 | EXISTING |
| 5 | NEW |
| 6 | RESURRECT |

**daily_pay:**

| user_id | paid |
|--------:|-----:|
| 1 | 150.5 |
| 2 | 200 |
| 3 | 175.25 |
| 5 | 125.75 |
| 7 | 100 |
| 8 | 250 |

**Output:**

| user_id | new_status |
|--------:|------------|
| 1 | EXISTING |
| 2 | EXISTING |
| 3 | RESURRECT |
| 4 | CHURN |
| 5 | EXISTING |
| 6 | CHURN |
| 7 | NEW |
| 8 | NEW |

**Explanation:** Users 1, 2, and 5 made payments while previously being active, so they become `EXISTING`. User 3 was previously `CHURN` and made a payment, so becomes `RESURRECT`. Users 4 and 6 did not make a payment and therefore become `CHURN`. Users 7 and 8 have no previous advertiser record but made a payment, so they are classified as `NEW`.

### Constraints

- Payment with prior `CHURN` results in `RESURRECT`.
- Payment with prior `NEW`, `EXISTING`, or `RESURRECT` results in `EXISTING`.
- No payment with prior `NEW`, `EXISTING`, or `RESURRECT` results in `CHURN`.
- Payment with no prior advertiser record results in `NEW`.
- Order the results by `user_id` in ascending order.

In [0]:
advertiser_data=[(1,"NEW"),(2,"EXISTING"),(3,"CHURN"),(4,"EXISTING"),(5,"NEW"),(6,"RESURRECT")]
advertiser_df=spark.createDataFrame(advertiser_data,["user_id","status"])
display(advertiser_df)

daily_pay_data=[(1,150.5),(2,200.0),(3,175.25),(5,125.75),(7,100.0),(8,250.0)]
daily_pay_df=spark.createDataFrame(daily_pay_data,["user_id","paid"])
display(daily_pay_df)

joined_df = advertiser_df.join(daily_pay_df,on="user_id",how="full_outer")


# user_id, status, paid

makes_payment = col("paid").isNotNull()
in_list = col("status").isin('NEW', 'EXISTING', 'RESURRECT')

output_df = (
joined_df.withColumn("new_status", 
    when( ((col("status") == 'CHURN') & (makes_payment)), "RESURRECT").
    when( ((in_list) & (makes_payment)), "EXISTING"  ).
    when(((in_list) & ~(makes_payment)), "CHURN").
    when(  ((col("status").isNull()) & (col("paid").isNotNull())), "NEW")
)
.select("user_id", "new_status")
.orderBy("user_id")
)


display(output_df)


user_id,status
1,NEW
2,EXISTING
3,CHURN
4,EXISTING
5,NEW
6,RESURRECT


user_id,paid
1,150.5
2,200.0
3,175.25
5,125.75
7,100.0
8,250.0


user_id,new_status
1,EXISTING
2,EXISTING
3,RESURRECT
4,CHURN
5,EXISTING
6,CHURN
7,NEW
8,NEW


%md
## Handle Data Skew in Joins

**Difficulty:** Hard

### Problem

A data engineering team wants to analyze user engagement by account type. Each event belongs to a user, but only events with a matching user record should be included in the analysis.

For each account type, report:
- The total number of matched events.
- The number of distinct users who generated at least one matched event.
- The average number of events per user, calculated as `total_events / unique_users` and rounded to 2 decimal places.

**Schema columns:** `events.event_id`, `events.user_id`, `events.event_type`, `events.event_date`, `users.user_id`, `users.user_name`, `users.account_type`

**Output columns:** `account_type`, `total_events`, `unique_users`, `events_per_user`

Order the result by `total_events` in descending order.

### Examples

#### Example 1

**Input:**

**events:**

| event_id | user_id | event_type | event_date |
|---------:|--------:|------------|------------|
| 1 | 1 | purchase | 2024-01-05 |
| 2 | 1 | view | 2024-01-06 |
| 3 | 1 | click | 2024-01-09 |
| 4 | 4 | view | 2024-01-11 |
| 5 | 4 | share | 2024-01-12 |
| 6 | 7 | purchase | 2024-01-14 |
| 7 | 7 | view | 2024-01-18 |
| 8 | 2 | click | 2024-01-08 |
| 9 | 2 | purchase | 2024-01-15 |
| 10 | 3 | view | 2024-01-20 |
| 11 | 99 | click | 2024-01-22 |

**users:**

| user_id | user_name | account_type |
|--------:|-----------|--------------|
| 1 | User_1 | Premium |
| 2 | User_2 | Basic |
| 3 | User_3 | Standard |
| 4 | User_4 | Premium |
| 7 | User_7 | Premium |

**Output:**

| account_type | total_events | unique_users | events_per_user |
|--------------|-------------:|-------------:|----------------:|
| Premium | 7 | 3 | 2.33 |
| Basic | 2 | 1 | 2.00 |
| Standard | 1 | 1 | 1.00 |

**Explanation:** Only events with matching users are included in the aggregation. Event 11 is excluded because its `user_id` does not exist in the `users` table. Premium users generate 7 events across 3 distinct users, resulting in `7 / 3 = 2.33` events per user.

### Constraints

- Include only events whose `user_id` exists in the `users` table.
- `unique_users` counts distinct users with at least one matched event.
- Round `events_per_user` to 2 decimal places.
- Sort the result by `total_events` in descending order.

In [0]:
events_data=[(1,1,"purchase","2024-01-05"),(2,1,"view","2024-01-06"),(3,1,"click","2024-01-09"),(4,4,"view","2024-01-11"),(5,4,"share","2024-01-12"),(6,7,"purchase","2024-01-14"),(7,7,"view","2024-01-18"),(8,2,"click","2024-01-08"),(9,2,"purchase","2024-01-15"),(10,3,"view","2024-01-20"),(11,99,"click","2024-01-22")]
events_df=spark.createDataFrame(events_data,["event_id","user_id","event_type","event_date"])
display(events_df)

users_data=[(1,"User_1","Premium"),(2,"User_2","Basic"),(3,"User_3","Standard"),(4,"User_4","Premium"),(7,"User_7","Premium")]
users_df=spark.createDataFrame(users_data,["user_id","user_name","account_type"])
display(users_df)

joined_df = (
events_df.alias("e")
    .join(users_df.alias("u"), on=col("e.user_id") == col("u.user_id"), how="inner")
)

# account_type	total_events	unique_users	events_per_user
output_df = (
joined_df.groupBy("u.account_type").agg(
    count("e.event_id").alias("total_events"),
    countDistinct("u.user_id").alias("unique_users")
)
.withColumn("events_per_user", round(col("total_events") / col("unique_users"), 2))
.orderBy(col("total_events").desc())
)


display(output_df)


event_id,user_id,event_type,event_date
1,1,purchase,2024-01-05
2,1,view,2024-01-06
3,1,click,2024-01-09
4,4,view,2024-01-11
5,4,share,2024-01-12
6,7,purchase,2024-01-14
7,7,view,2024-01-18
8,2,click,2024-01-08
9,2,purchase,2024-01-15
10,3,view,2024-01-20


user_id,user_name,account_type
1,User_1,Premium
2,User_2,Basic
3,User_3,Standard
4,User_4,Premium
7,User_7,Premium


account_type,total_events,unique_users,events_per_user
Premium,7,3,2.33
Basic,2,1,2.0
Standard,1,1,1.0


%md
## Complex Join with Multiple Conditions

**Difficulty:** Hard

### Problem

An online store applies promotional discounts based on a product's category, the order's region, and whether the order date falls within the promotion's validity period.

For every order line item, determine the applicable discount and calculate the final price after the discount.

A discount applies only if:
- The product category matches the promotion category.
- The order region matches the promotion region.
- The order date falls between the promotion's `valid_from` and `valid_to` dates (inclusive).

If no promotion matches, the discount percentage is `0`.

The final price is calculated as:

`quantity × unit_price × (1 - discount_pct / 100)`

rounded to 2 decimal places.

**Schema columns:** `discounts.category`, `discounts.region`, `discounts.discount_pct`, `discounts.valid_from`, `discounts.valid_to`, `order_items.order_id`, `order_items.product_id`, `order_items.quantity`, `order_items.unit_price`, `orders.order_id`, `orders.customer_id`, `orders.order_date`, `orders.region`, `products.product_id`, `products.product_name`, `products.category`

**Output columns:** `order_id`, `product_name`, `category`, `region`, `quantity`, `unit_price`, `discount_pct`, `final_price`

Order the result by `order_id` in ascending order, then by `product_name` in ascending order.

### Examples

#### Example 1

**Input:**

**discounts:**

| category | region | discount_pct | valid_from | valid_to |
|----------|--------|-------------:|------------|------------|
| Electronics | North | 10 | 2024-01-01 | 2024-02-28 |
| Furniture | North | 15 | 2024-01-15 | 2024-03-31 |
| Electronics | South | 5 | 2024-02-01 | 2024-03-31 |

**orders:**

| order_id | customer_id | order_date | region |
|---------:|-------------|------------|--------|
| 1001 | C101 | 2024-01-15 | North |
| 1002 | C102 | 2024-01-20 | South |
| 1003 | C103 | 2024-02-10 | West |

**order_items:**

| order_id | product_id | quantity | unit_price |
|---------:|------------|---------:|-----------:|
| 1001 | P001 | 1 | 1000 |
| 1001 | P003 | 1 | 300 |
| 1002 | P001 | 1 | 1000 |
| 1003 | P002 | 2 | 50 |

**products:**

| product_id | product_name | category |
|------------|--------------|----------|
| P001 | Laptop | Electronics |
| P002 | Mouse | Electronics |
| P003 | Desk | Furniture |

**Output:**

| order_id | product_name | category | region | quantity | unit_price | discount_pct | final_price |
|---------:|--------------|----------|--------|---------:|-----------:|-------------:|------------:|
| 1001 | Desk | Furniture | North | 1 | 300 | 15 | 255.00 |
| 1001 | Laptop | Electronics | North | 1 | 1000 | 10 | 900.00 |
| 1002 | Laptop | Electronics | South | 1 | 1000 | 0 | 1000.00 |
| 1003 | Mouse | Electronics | West | 2 | 50 | 0 | 100.00 |

**Explanation:** Order 1001 qualifies for both applicable North region promotions based on product category and order date. Order 1002 does not receive the South Electronics promotion because the order date is before the promotion starts. Order 1003 has no matching promotion for the West region, so no discount is applied.

### Constraints

- A discount applies only when category and region both match and the order date falls within the promotion period (inclusive).
- If no promotion matches, `discount_pct` is `0`.
- Calculate `final_price` as `quantity × unit_price × (1 - discount_pct / 100)` and round to 2 decimal places.
- Order the result by `order_id` ascending, then `product_name` ascending.

In [0]:
discounts_data=[("Electronics","North",10,"2024-01-01","2024-02-28"),("Furniture","North",15,"2024-01-15","2024-03-31"),("Electronics","South",5,"2024-02-01","2024-03-31")]
discounts_df=spark.createDataFrame(discounts_data,["category","region","discount_pct","valid_from","valid_to"])
display(discounts_df)

orders_data=[(1001,"C101","2024-01-15","North"),(1002,"C102","2024-01-20","South"),(1003,"C103","2024-02-10","West")]
orders_df=spark.createDataFrame(orders_data,["order_id","customer_id","order_date","region"])
display(orders_df)

order_items_data=[(1001,"P001",1,1000),(1001,"P003",1,300),(1002,"P001",1,1000),(1003,"P002",2,50)]
order_items_df=spark.createDataFrame(order_items_data,["order_id","product_id","quantity","unit_price"])
display(order_items_df)

products_data=[("P001","Laptop","Electronics"),("P002","Mouse","Electronics"),("P003","Desk","Furniture")]
products_df=spark.createDataFrame(products_data,["product_id","product_name","category"])
display(products_df)

# order_id	product_name	category	region	quantity	unit_price	discount_pct	final_price

discount_condition = (col("d.category") == col("p.category")) & (col("d.region") == col("o.region")) & (col("o.order_date") >= col("d.valid_from")) & (col("o.order_date") <= col("d.valid_to"))

joined_df = (
orders_df.alias("o")
    .join(order_items_df.alias("oi"), on=col("o.order_id") == col("oi.order_id"), how="left")
    .join(products_df.alias("p"), on=col("oi.product_id") == col("p.product_id"), how="left")
    .join(discounts_df.alias("d"), on=(discount_condition), how="left")
    .select("o.order_id", "p.product_name",	"p.category", "o.region", "oi.quantity", "oi.unit_price", "d.discount_pct")
    .fillna(0, subset=["discount_pct"])
)

# quantity × unit_price × (1 - discount_pct / 100)

output_df = (
joined_df.withColumn("final_price", 
    round(col("quantity")  * col("unit_price") * (1 - col("discount_pct") / 100), 2))
    .orderBy("order_id", "product_name")
)

display(output_df)


category,region,discount_pct,valid_from,valid_to
Electronics,North,10,2024-01-01,2024-02-28
Furniture,North,15,2024-01-15,2024-03-31
Electronics,South,5,2024-02-01,2024-03-31


order_id,customer_id,order_date,region
1001,C101,2024-01-15,North
1002,C102,2024-01-20,South
1003,C103,2024-02-10,West


order_id,product_id,quantity,unit_price
1001,P001,1,1000
1001,P003,1,300
1002,P001,1,1000
1003,P002,2,50


product_id,product_name,category
P001,Laptop,Electronics
P002,Mouse,Electronics
P003,Desk,Furniture


order_id,product_name,category,region,quantity,unit_price,discount_pct,final_price
1001,Desk,Furniture,North,1,300,15,255.0
1001,Laptop,Electronics,North,1,1000,10,900.0
1002,Laptop,Electronics,South,1,1000,0,1000.0
1003,Mouse,Electronics,West,2,50,0,100.0
